In [ ]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
# from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
from shapely.geometry import shape 
import json 
from shapely import wkt 
from shapely.geometry import Point
from shapely.geometry import box
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import matplotlib
from statsmodels.tsa.seasonal import seasonal_decompose

import glob
import os
import csv
import ast

from scipy.stats import chi2_contingency
from math import sqrt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import seaborn as sns
from matplotlib.ticker import PercentFormatter
import matplotlib.lines as mlines

In [ ]:
from pathlib import Path

DATA_DIR  = Path('../files')
PLOTS_DIR = Path('../outputs/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
(PLOTS_DIR / 'supplementary').mkdir(exist_ok=True)
(PLOTS_DIR / 'spatial').mkdir(exist_ok=True)


In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Reading Temp & Precip (6km) 

In [ ]:
temp_all = pd.read_csv(DATA_DIR / 'processed_weather_data/temp_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])
temp_all['time'] = pd.to_datetime(temp_all['time'])

precip_all = pd.read_csv(DATA_DIR / 'processed_weather_data/precip_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])
precip_all['time'] = pd.to_datetime(precip_all['time'])

### Wind (6km) 

In [ ]:
hourly_wind_df = pd.read_csv(DATA_DIR / 'processed_weather_data/ea_hourly_wind_avg_6km.csv').drop(columns=['Unnamed: 0'])
hourly_wind_df['Timestamp'] = pd.to_datetime(hourly_wind_df['Timestamp'])
hourly_wind_df = hourly_wind_df.rename(columns = {'Timestamp':'time'})
hourly_wind_df['Date'] = hourly_wind_df['time'].dt.date

### Lightning (6km) 

In [ ]:
hourly_lightning_df = pd.read_csv(DATA_DIR / 'processed_weather_data/ea_hourly_lightning_avg_6km_buffer_aligned.csv').drop(columns=['Unnamed: 0'])
hourly_lightning_df['Timestamp'] = pd.to_datetime(hourly_lightning_df['Timestamp'])
hourly_lightning_df = hourly_lightning_df.rename(columns = {'Timestamp':'time'})
hourly_lightning_df['Date'] = hourly_lightning_df['time'].dt.date

### Extreme Definitions 

### Temp 

In [ ]:
# temp_all['Temp'].quantile(0.95)

In [ ]:
temp_all['Date'] = temp_all['time'].dt.floor('D')

# Hot hour flag
temp_all['Hot_Hour'] = temp_all['Temp'] > 32

# Group by EA and Date, and count Hot_Hour sum
daily_hot_hours = (
    temp_all.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Hot_Hour': 'sum'})
)

daily_hot_hours = daily_hot_hours.rename(columns={'Hot_Hour': 'Num_Hot_Hours'})

### Precip 

In [ ]:
precip_all['Date'] = precip_all['time'].dt.floor('D')  

# Group by EA and Date, sum Precip
daily_precip = (
    precip_all.groupby(['ea_code9ch', 'Date'], as_index=False)
      .agg({'Precip': 'sum'})
)

### Wind 

In [ ]:
# Windy hour flag
hourly_wind_df['Windy_Hour'] = hourly_wind_df['Wind Gusts (m/s)'] > 5.93

# Group by EA and Date, and count Windy_Hour sum
daily_windy_hours = (
    hourly_wind_df.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Windy_Hour': 'sum'})
)

daily_windy_hours = daily_windy_hours.rename(columns={'Windy_Hour': 'Num_Windy_Hours'})

### Lightning 

In [ ]:
daily_lightning_per_ea = hourly_lightning_df.groupby(['ea_code9ch', 'Date'])['Lightning Events'].sum().reset_index()

## Percentiles 

In [ ]:
### 90th pct 
hot_hrs_90_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.90)
temp_90_thresh = temp_all['Temp'].quantile(0.90)

precip_90_thresh = daily_precip['Precip'].quantile(0.90)
lightning_90_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.90)

windy_hrs_90_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.90)
wind_90_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.90)


### 95th pct 

hot_hrs_95_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.95)
temp_95_thresh = temp_all['Temp'].quantile(0.95)
precip_95_thresh = daily_precip['Precip'].quantile(0.95)
lightning_95_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.95)
windy_hrs_95_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.95)
wind_95_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.95)

### 99th pct 
hot_hrs_99_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.99)
temp_99_thresh = temp_all['Temp'].quantile(0.99)
precip_99_thresh = daily_precip['Precip'].quantile(0.99)
lightning_99_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.99)
windy_hrs_99_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.99)
wind_99_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.99)

#### Dictionary for percentiles 

In [ ]:
# Percentile Threshold dictionaries 
temp_thresh_dict = {
    '90': temp_90_thresh.round(2),
    '95': temp_95_thresh.round(2),
    '99': temp_99_thresh.round(2)
}

hot_hrs_thresh_dict = {
    '90': hot_hrs_90_thresh.round(2),
    '95': hot_hrs_95_thresh.round(2),
    '99': hot_hrs_99_thresh.round(2)
}

precip_thresh_dict = {
    '90': precip_90_thresh.round(2),
    '95': precip_95_thresh.round(2),
    '99': precip_99_thresh.round(2)
}

wind_thresh_dict = {
    '90': wind_90_thresh.round(2),
    '95': wind_95_thresh.round(2),
    '99': wind_99_thresh.round(2)
}

windy_hrs_thresh_dict = {
    '90': windy_hrs_90_thresh.round(2),
    '95': windy_hrs_95_thresh.round(2),
    '99': windy_hrs_99_thresh.round(2)
}

lightning_thresh_dict = {
    '90': 1,   # at least 1 lightning strike 
    '95': lightning_95_thresh.round(2),
    '99': lightning_99_thresh.round(2)
}

### Read EAs and CESI level 

In [ ]:
eas_n_cesi_level = pd.read_csv(DATA_DIR / 'miscellaneous/eas_cesi_level_214_eas_rev1.csv')

### EA Names mapped  

In [ ]:
ea_names_mapped = pd.read_csv(DATA_DIR / 'miscellaneous/ea_names_mapped.csv')

ea_names_mapped = ea_names_mapped[['ea_code9ch', 'LOC_NAME', 'BASE_NAM', 'geometry']]

### EAs n Sites  (6km) 

In [ ]:
merged_eas_sites = pd.read_csv(DATA_DIR / 'miscellaneous/ea_site_list_6km_buffer.csv')
merged_eas_sites = merged_eas_sites[['ea_code9ch', 'Intersecting_Sites']]

# Convert the string representation of lists to actual lists
merged_eas_sites['Intersecting_Sites'] = merged_eas_sites['Intersecting_Sites'].apply(ast.literal_eval)

In [ ]:
# merged_eas_sites['ea_code9ch'].nunique()

In [ ]:
filtered_eas_sites_copy = merged_eas_sites

In [ ]:
# EA -> 30200014, 30500020 

In [ ]:
def prepare_hourly_weather_df_TPLW(
    ea_row, 
    temp_df, 
    precip_df, 
    lightning_df, 
    wind_df, 
):
    ea = ea_row['ea_code9ch']
    site_list = [int(s) for s in ea_row['Intersecting_Sites']]
    all_sites_dfs = []

    for site_id in site_list:
        # --- Temperature ---
        temp_filt = (
            temp_df[temp_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            .copy()
        )
        if 'Hot_Hour' in temp_filt.columns:
            temp_filt = temp_filt.drop(columns=['Hot_Hour'])
        temp_filt['time'] = pd.to_datetime(temp_filt['time'])

        # --- Precipitation ---
        precip_filt = (
            precip_df[precip_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Precip']]
        )
        precip_filt['time'] = pd.to_datetime(precip_filt['time'])

        # Merge temp + precip
        merged = temp_filt.merge(precip_filt, on='time', how='outer')

        # --- Lightning ---
        lightning_filt = (
            lightning_df[lightning_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Lightning Events']]
        )
        merged = merged.merge(lightning_filt, on='time', how='outer')
        merged['Lightning Events'] = merged['Lightning Events'].fillna(0)

        # --- Wind ---
        wind_filt = (
            wind_df[wind_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Wind Gusts (m/s)']]
        )
        merged = merged.merge(wind_filt, on='time', how='inner')

        # Add site_id
        merged['site_id'] = site_id

        all_sites_dfs.append(merged)

    # Concatenate all sites
    df_all = pd.concat(all_sites_dfs, ignore_index=True)

    # Aggregate by timestamp 
    agg_dict = {
        'Temp': 'mean',  # average across sites
        'Precip': 'mean',
        'Lightning Events': 'mean',
        'Wind Gusts (m/s)': 'mean',
        'site_id': lambda x: ','.join(map(str, x))  # list all contributing sites
    }

    df_agg = df_all.groupby('time', as_index=False).agg(agg_dict)

    # Add EA code
    df_agg['ea_code9ch'] = ea

    # Optional: reorder columns
    df_agg = df_agg[
        ['time', 'ea_code9ch', 'site_id', 'Temp', 'Precip', 
         'Lightning Events', 'Wind Gusts (m/s)']
    ]

    return df_agg

## Remove sites with > `10%` missing data & less than 24 months 

In [ ]:
sites_to_omit = pd.read_csv(DATA_DIR / 'miscellaneous/complete_site_removal_df.csv')['site_id'].to_list()

### Resample hourly weather data -> Daily 

In [ ]:
filtered_eas_sites_copy_r1 = merged_eas_sites.copy()

In [ ]:
# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_weather_df_TPLW(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

In [ ]:
merged_hourly_data_global = merged_hourly_data_global.loc[
    ~merged_hourly_data_global['site_id'].isin(sites_to_omit)
]

In [ ]:
### Filter 214 EAs 

tplw_eas = pd.read_csv(DATA_DIR / 'miscellaneous/unique_ea_codes_TPLW.csv')

list_EAs = tplw_eas['ea_code9ch'].unique().tolist()

hourly_global_test = merged_hourly_data_global[merged_hourly_data_global['ea_code9ch'].isin(list_EAs)].reset_index(drop=True)

In [ ]:
hourly_global_test['ea_code9ch'].nunique()

In [ ]:
# 1. Convert to string (just in case)
hourly_global_test['site_id'] = hourly_global_test['site_id'].astype(str)

# 2. Split into list
hourly_global_test['site_id'] = hourly_global_test['site_id'].str.split(',')

# 3. Explode into separate rows
hourly_global_test = hourly_global_test.explode('site_id')

# 4. Clean up (remove spaces + convert to int)
hourly_global_test['site_id'] = hourly_global_test['site_id'].str.strip().astype(int)

### Filter Weather data 

In [ ]:
hourly_temperature = hourly_global_test.drop(columns = ['Precip', 'Lightning Events', 'Wind Gusts (m/s)']).reset_index(drop=True)

### save to file 

In [ ]:
# hourly_temperature.to_csv(DATA_DIR / 'miscellaneous/hourly_temp_sites_of_interest.csv')

## *** Regression Analysis 

### read hourly voltage, EA <-> CESI Mapping & Merge  

### All sensors 

In [ ]:
hourly_voltage_22 = pd.read_csv(DATA_DIR / 'processed_undervolt_data/undervolt_hourly_voltages_df_22.csv')
hourly_voltage_23 = pd.read_csv(DATA_DIR / 'processed_undervolt_data/undervolt_hourly_voltages_df_23.csv')

hourly_voltage = pd.concat([hourly_voltage_22, hourly_voltage_23], axis=0, ignore_index=True)
hourly_voltage['time'] = pd.to_datetime(hourly_voltage['time'])

### sensor of interest 

In [ ]:
sensor_of_interest_list = pd.read_csv(DATA_DIR / 'miscellaneous/common_sensors_22_23_rev4.csv')['respondent_id'].unique().tolist()

hourly_voltage = hourly_voltage[hourly_voltage['sensor_id'].isin(sensor_of_interest_list)].reset_index(drop=True)

In [ ]:
## EA <-> CESI Mapping 
sensor_ea_mapping = pd.read_csv(DATA_DIR / 'miscellaneous/sensor_ea_mapping_rev1.csv').drop(columns=['Unnamed: 0'])

## Merge 
hourly_voltage = hourly_voltage.merge(sensor_ea_mapping, on = ['sensor_id', 'site_id'], how = 'left')

In [ ]:
# hourly_voltage['sensor_id'].nunique()

In [ ]:
# hourly_voltage['site_id'].nunique()

In [ ]:
hourly_voltage

### sanity check -> double checking that all sites in `hourly_voltage` exist in `hourly_temperature` 

In [ ]:
voltage_sites = set(hourly_voltage['site_id'].unique())
temperature_sites = set(hourly_temperature['site_id'].unique())

missing_in_temperature = voltage_sites - temperature_sites

print(f"Number of voltage sites: {len(voltage_sites)}")
print(f"Number of temperature sites: {len(temperature_sites)}")
print(f"Sites missing in hourly_temperature: {len(missing_in_temperature)}")

print(sorted(missing_in_temperature))

### Combine weather & voltage 

In [ ]:
hourly_volt_n_temp = hourly_voltage.merge(hourly_temperature, on = ['time', 'site_id', 'ea_code9ch'], how = 'left')
hourly_volt_n_temp = hourly_volt_n_temp[['time', 'ea_code9ch', 'site_id', 'sensor_id', 'voltage', 'Temp', 'cesi_level']]
hourly_volt_n_temp['Temp'] = hourly_volt_n_temp['Temp'].round(3)

### Remove NaNs 

In [ ]:
hourly_volt_n_temp = hourly_volt_n_temp.dropna().reset_index(drop=True)

## Pooled Workflow  -> all sensors 

### Pooled Regression -> all sensors 

In [ ]:
## per sensor 
## rolling mean 

In [ ]:
import pandas as pd
import statsmodels.api as sm

def run_voltage_temp_ols(
    df,
    time_col='time',
    temp_col='Temp',
    voltage_col='voltage',
    sensor_col='sensor_id',
    use_hac=False,
    hac_lags=None,
    drop_first=True,
    include_sensor_fe=False,
    rolling_hours=None
):
    """
    Run OLS of voltage on temperature, controlling for month, hour,
    and optionally sensor fixed effects.

    If rolling_hours is provided, uses rolling mean temperature over that
    number of hours instead of raw temperature.
    """

    df = df.copy()

    # Ensure datetime and sort
    df[time_col] = pd.to_datetime(df[time_col])

    sort_cols = [time_col]
    if include_sensor_fe and sensor_col in df.columns:
        sort_cols = [sensor_col, time_col]

    df = df.sort_values(sort_cols)

    # Create time features
    df['month'] = df[time_col].dt.month
    df['hour'] = df[time_col].dt.hour

    # Temperature variable
    if rolling_hours is not None:
        temp_roll_col = f'{temp_col}_{rolling_hours}hr_avg'

        if include_sensor_fe:
            df[temp_roll_col] = (
                df.groupby(sensor_col)[temp_col]
                  .rolling(window=rolling_hours, min_periods=rolling_hours)
                  .mean()
                  .reset_index(level=0, drop=True)
            )
        else:
            df[temp_roll_col] = (
                df[temp_col]
                .rolling(window=rolling_hours, min_periods=rolling_hours)
                .mean()
            )

        temp_used = temp_roll_col
    else:
        temp_used = temp_col

    # Drop rows with missing rolling values
    df = df.dropna(subset=[temp_used, voltage_col]).copy()

    # Columns to include in design matrix
    cols = [temp_used, 'month', 'hour']

    if include_sensor_fe:
        cols.append(sensor_col)

    # Create design matrix
    X = pd.get_dummies(
        df[cols],
        columns=['month', 'hour'] + ([sensor_col] if include_sensor_fe else []),
        drop_first=drop_first
    )

    X = X.astype(float)
    X = sm.add_constant(X)

    y = df[voltage_col].astype(float)

    # Fit model
    if use_hac:
        if hac_lags is None:
            raise ValueError("hac_lags must be provided when use_hac=True")

        model = sm.OLS(y, X).fit(
            cov_type='HAC',
            cov_kwds={'maxlags': hac_lags}
        )
    else:
        model = sm.OLS(y, X).fit()

    return model

### WITH sensor fixed effects (this means I'm controlling for the sensor) 

In [ ]:
# overall_regression_with_sensor_fe = model = run_voltage_temp_ols(
#     hourly_volt_n_temp.copy(),
#     use_hac = False,
#     hac_lags = None,
#     drop_first = True, 
#     include_sensor_fe = True, 
#     rolling_hours = None
# )

In [ ]:
# overall_regression_with_sensor_fe.summary()

### super interesting results here (if I decide to include this) 

### Adjust for clustered sensor observations --> do this for the OVERALL regression  

In [ ]:
import pandas as pd
import statsmodels.api as sm

def run_voltage_temp_ols_clustered(
    df,
    time_col='time',
    temp_col='Temp',
    voltage_col='voltage',
    sensor_col='sensor_id',
    drop_first=True
):
    """
    OLS of voltage on temperature, controlling for month and hour,
    with clustered standard errors by sensor.

    NO sensor fixed effects.
    """

    df = df.copy()

    # Ensure datetime
    df[time_col] = pd.to_datetime(df[time_col])

    # Time features
    df['month'] = df[time_col].dt.month
    df['hour'] = df[time_col].dt.hour

    # Drop missing
    df = df.dropna(subset=[temp_col, voltage_col, sensor_col]).copy()

    # Design matrix (month + hour controls only)
    X = pd.get_dummies(
        df[[temp_col, 'month', 'hour']],
        columns=['month', 'hour'],
        drop_first=drop_first
    )

    X = X.astype(float)
    X = sm.add_constant(X)

    y = df[voltage_col].astype(float)

    # Clustered SE by sensor
    model = sm.OLS(y, X).fit(
        cov_type='cluster',
        cov_kwds={'groups': df[sensor_col]}
    )

    return model

In [ ]:
## updated -> hour & month fe with flags 

In [ ]:
import pandas as pd
import statsmodels.api as sm

def run_voltage_temp_ols_clustered(
    df,
    time_col='time',
    temp_col='Temp',
    voltage_col='voltage',
    sensor_col='sensor_id',
    drop_first=True,
    control_months=True,
    control_hours=True
):
    """
    OLS of voltage on temperature, controlling for month and/or hour,
    with clustered standard errors by sensor.
    NO sensor fixed effects.

    Parameters
    ----------
    control_months : bool
        If True, include month dummies as controls.
    control_hours : bool
        If True, include hour dummies as controls.
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    df['month'] = df[time_col].dt.month
    df['hour'] = df[time_col].dt.hour
    df = df.dropna(subset=[temp_col, voltage_col, sensor_col]).copy()

    cols = [temp_col]
    dummy_cols = []
    if control_months:
        cols.append('month')
        dummy_cols.append('month')
    if control_hours:
        cols.append('hour')
        dummy_cols.append('hour')

    X = df[cols].copy()
    if dummy_cols:
        X = pd.get_dummies(X, columns=dummy_cols, drop_first=drop_first)

    X = X.astype(float)
    X = sm.add_constant(X)
    y = df[voltage_col].astype(float)

    model = sm.OLS(y, X).fit(
        cov_type='cluster',
        cov_kwds={'groups': df[sensor_col]}
    )
    return model

### No FE 

In [ ]:
overall_regression_with_sensor_clustering = model = run_voltage_temp_ols_clustered(
    hourly_volt_n_temp.copy(),
    drop_first = True, 
    control_months=False,
    control_hours=False
)

In [ ]:
overall_regression_with_sensor_clustering.summary()

### + Month FE 

In [ ]:
overall_regression_with_sensor_clustering = model = run_voltage_temp_ols_clustered(
    hourly_volt_n_temp.copy(),
    drop_first = True, 
    control_months=True,
    control_hours=False
)

In [ ]:
overall_regression_with_sensor_clustering.summary()

### + Month & Hour FE 

In [ ]:
overall_regression_with_sensor_clustering = model = run_voltage_temp_ols_clustered(
    hourly_volt_n_temp.copy(),
    drop_first = True, 
    control_months=True,
    control_hours=True
)

In [ ]:
overall_regression_with_sensor_clustering.summary()

### plot month effects 

In [ ]:
import pandas as pd

# Extract model outputs
params = overall_regression_with_sensor_clustering.params
std_err = overall_regression_with_sensor_clustering.bse
pvals = overall_regression_with_sensor_clustering.pvalues

# Filter month terms
month_coefs = params.filter(like='month_')
month_se = std_err.filter(like='month_')
month_pvals = pvals.filter(like='month_')

# Build dataframe
month_effects_df = pd.DataFrame({
    'month': month_coefs.index,
    'coef': month_coefs.values,
    'se': month_se.values,
    'p_value': month_pvals.values
})

# Extract month number
month_effects_df['month_num'] = month_effects_df['month'].str.extract(r'(\d+)').astype(int)

# Sort
month_effects_df = month_effects_df.sort_values('month_num')

# Round numeric columns (except month_num)
month_effects_df[['coef', 'se', 'p_value']] = month_effects_df[['coef', 'se', 'p_value']].round(3)

In [ ]:
df = month_effects_df.copy()

In [ ]:
def plot_month_effects(
    df,
    ylim=None,
    show_error=True,
    sig_level=0.05,
    legend_loc='best',
    legend_bbox=None,
    figsize=(10, 5)
):
    import matplotlib.pyplot as plt
    import calendar
    import numpy as np
    import matplotlib.lines as mlines

    plt.figure(figsize=figsize)

    # Plot coefficients
    plt.plot(df['month_num'], df['coef'], marker='o')

    # Error bars
    if show_error:
        plt.errorbar(
            df['month_num'],
            df['coef'],
            yerr=1.96 * df['se'],
            fmt='none',
            color='gray',
            linewidth=1,
            alpha=0.7
        )

    # Month labels
    month_labels = [calendar.month_abbr[m] for m in df['month_num']]

    # Axis labels
    plt.xlabel("Month", fontsize=16, labelpad=15, fontweight='bold')
    plt.ylabel("Coefficients", fontsize=16, labelpad=15, fontweight='bold')

    # X ticks
    plt.xticks(df['month_num'], month_labels, fontsize=12)

    # Reference line
    plt.axhline(0, linestyle='--', color='r')

    # --- Y limits ---
    if ylim is None:
        max_val = np.max(np.abs(df['coef'] + 1.96 * df['se']))
        max_val = np.ceil(max_val * 2) / 2
        ylim = (-max_val, max_val)

    plt.ylim(ylim)

    # --- Y ticks every 0.5 ---
    y_ticks = np.arange(ylim[0], ylim[1] + 0.5, 0.5)
    plt.yticks(y_ticks, fontsize=12)

    # --- Significance stars ---
    y_offset = (ylim[1] - ylim[0]) * 0.03

    for _, row in df.iterrows():
        if row['p_value'] < sig_level:
            plt.text(
                row['month_num'],
                row['coef'] + y_offset,
                '*',
                ha='center',
                va='bottom',
                fontsize=12,
                color='black'
            )

    # --- Legend handles ---
    legend_handles = []

    if show_error:
        err_handle = mlines.Line2D(
            [], [],
            color='gray',
            linewidth=1,
            label='95% CI'
        )
        legend_handles.append(err_handle)

    star_handle = mlines.Line2D(
        [], [],
        color='black',
        marker='*',
        linestyle='None',
        markersize=10,
        label='p < 0.05'
    )
    legend_handles.append(star_handle)

    # --- Legend ---
    plt.legend(
        handles=legend_handles,
        fontsize=14,          # match hour plot
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    # Remove top and right spines
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_month_effects(
    month_effects_df,
    ylim=(-2.5, 2.5),
    legend_loc='upper left',
    legend_bbox=(0.9, 1),
    figsize=(10, 5)
)

### plot hour effects 

In [ ]:
import pandas as pd

# Extract model outputs
params = overall_regression_with_sensor_clustering.params
std_err = overall_regression_with_sensor_clustering.bse
pvals = overall_regression_with_sensor_clustering.pvalues

# Filter hour terms
hour_coefs = params.filter(like='hour_')
hour_se = std_err.filter(like='hour_')
hour_pvals = pvals.filter(like='hour_')

# Build dataframe
hour_effects_df = pd.DataFrame({
    'hour': hour_coefs.index,
    'coef': hour_coefs.values,
    'se': hour_se.values,
    'p_value': hour_pvals.values
})

# Extract hour number
hour_effects_df['hour_num'] = hour_effects_df['hour'].str.extract(r'(\d+)').astype(int)

# Sort
hour_effects_df = hour_effects_df.sort_values('hour_num')

# Round numeric columns except hour_num
hour_effects_df[['coef', 'se', 'p_value']] = (
    hour_effects_df[['coef', 'se', 'p_value']].round(3)
)

In [ ]:
def plot_hour_effects(
    df,
    ylim=None,
    show_error=True,
    shade_range=None,
    shade_label=None,
    legend_loc='best',
    legend_bbox=None,
    figsize=(12, 5),
    sig_level=0.05,
    use_p_value=True
):
    import matplotlib.pyplot as plt
    import numpy as np
    import matplotlib.patches as mpatches
    import matplotlib.lines as mlines

    plt.figure(figsize=figsize)

    # Plot coefficients
    plt.plot(df['hour_num'], df['coef'], marker='o')

    # Error bars
    if show_error:
        plt.errorbar(
            df['hour_num'],
            df['coef'],
            yerr=1.96 * df['se'],
            fmt='none',
            color='gray',
            linewidth=1,
            alpha=0.7
        )

    # Axis labels
    plt.xlabel("Hour of day", fontsize=16, labelpad=15, fontweight='bold')
    plt.ylabel("Coefficients", fontsize=16, labelpad=15, fontweight='bold')

    # X ticks
    plt.xticks(df['hour_num'], df['hour_num'], fontsize=12)

    # Reference line
    plt.axhline(0, linestyle='--', color='r')

    # Y limits
    if ylim is None:
        max_val = np.max(np.abs(df['coef'] + 1.96 * df['se']))
        max_val = np.ceil(max_val)
        ylim = (-max_val, max_val)

    plt.ylim(ylim)

    # Y ticks
    step = 1 if (ylim[1] - ylim[0]) <= 10 else 2
    y_ticks = np.arange(ylim[0], ylim[1] + step, step)
    plt.yticks(y_ticks, fontsize=12)

    # Significance stars
    has_p_value = use_p_value and ('p_value' in df.columns)

    if has_p_value:
        y_offset = (ylim[1] - ylim[0]) * 0.03

        for _, row in df.iterrows():
            if row['p_value'] < sig_level:
                plt.text(
                    row['hour_num'],
                    row['coef'] + y_offset,
                    '*',
                    ha='center',
                    va='bottom',
                    fontsize=10,
                    color='black'
                )

    # Shaded region
    if shade_range is not None:
        start, end = shade_range
        plt.axvspan(start, end, color='orange', alpha=0.15)

    # Legend handles
    legend_handles = []

    if shade_range is not None and shade_label is not None:
        legend_handles.append(
            mpatches.Patch(
                color='orange',
                alpha=0.15,
                label=shade_label
            )
        )

    if show_error:
        legend_handles.append(
            mlines.Line2D(
                [], [],
                color='gray',
                linewidth=1,
                label='95% CI'
            )
        )

    if has_p_value:
        legend_handles.append(
            mlines.Line2D(
                [], [],
                color='black',
                marker='*',
                linestyle='None',
                markersize=8,
                label=f'p < {sig_level}'
            )
        )

    if legend_handles:
        plt.legend(
            handles=legend_handles,
            fontsize=14,
            frameon=True,
            loc=legend_loc,
            bbox_to_anchor=legend_bbox
        )

    # Remove top and right spines
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_hour_effects(
    hour_effects_df,
    figsize=(13, 5),
    ylim=(-8, 8),
    shade_range=(17, 21),
    shade_label="Peak load period",
    legend_loc='upper left',
    legend_bbox=(0.9, 1)
)

### Vertically stacked plots 

In [ ]:
def plot_month_hour_effects(
    month_df,
    hour_df,
    month_ylim=None,
    hour_ylim=None,
    show_error=True,
    sig_level=0.05,
    shade_range=None,
    shade_label=None,
    legend_loc='best',
    legend_bbox=None,
    figsize=(12, 10),
    save_path=None,
    dpi=300
):
    import matplotlib.pyplot as plt
    import matplotlib.lines as mlines
    import matplotlib.patches as mpatches
    import calendar
    import numpy as np

    fig, (ax_month, ax_hour) = plt.subplots(
        2, 1,
        figsize=figsize,
        sharex=False
    )

    # -------------------------
    # MONTH PANEL
    # -------------------------
    ax_month.plot(month_df['month_num'], month_df['coef'], marker='o')

    if show_error:
        ax_month.errorbar(
            month_df['month_num'], month_df['coef'],
            yerr=1.96 * month_df['se'],
            fmt='none', color='gray', linewidth=1, alpha=0.7
        )

    month_labels = [calendar.month_abbr[m] for m in month_df['month_num']]
    ax_month.set_xticks(month_df['month_num'])
    ax_month.set_xticklabels(month_labels, fontsize=12)
    ax_month.tick_params(axis='y', labelsize=12)
    ax_month.set_xlabel("Month", fontsize=16, labelpad=15, fontweight='bold')
    ax_month.axhline(0, linestyle='--', color='r', label='Zero effect')

    if month_ylim is None:
        max_val = np.max(np.abs(month_df['coef'] + 1.96 * month_df['se']))
        max_val = np.ceil(max_val * 2) / 2
        month_ylim = (-max_val, max_val)
    ax_month.set_ylim(month_ylim)
    y_ticks = np.arange(month_ylim[0], month_ylim[1] + 0.5, 0.5)
    ax_month.set_yticks(y_ticks)

    y_offset = (month_ylim[1] - month_ylim[0]) * 0.03
    for _, row in month_df.iterrows():
        if row['p_value'] < sig_level:
            ax_month.text(
                row['month_num'], row['coef'] + y_offset,
                '*', ha='center', va='bottom', fontsize=12, color='black'
            )

    ax_month.spines['top'].set_visible(False)
    ax_month.spines['right'].set_visible(False)
    ax_month.text(
        -0.07, 1.0, 'a', transform=ax_month.transAxes,
        fontsize=16, fontweight='bold', va='top'
    )

    # -------------------------
    # HOUR PANEL
    # -------------------------
    ax_hour.plot(hour_df['hour_num'], hour_df['coef'], marker='o')

    if show_error:
        ax_hour.errorbar(
            hour_df['hour_num'], hour_df['coef'],
            yerr=1.96 * hour_df['se'],
            fmt='none', color='gray', linewidth=1, alpha=0.7
        )

    ax_hour.set_xticks(hour_df['hour_num'])
    ax_hour.set_xticklabels(hour_df['hour_num'], fontsize=12)
    ax_hour.tick_params(axis='y', labelsize=12)
    ax_hour.set_xlabel("Hour of day", fontsize=16, labelpad=15, fontweight='bold')
    ax_hour.axhline(0, linestyle='--', color='r', label='_nolegend_')

    if hour_ylim is None:
        max_val = np.max(np.abs(hour_df['coef'] + 1.96 * hour_df['se']))
        max_val = np.ceil(max_val)
        hour_ylim = (-max_val, max_val)
    ax_hour.set_ylim(hour_ylim)
    step = 1 if (hour_ylim[1] - hour_ylim[0]) <= 10 else 2
    y_ticks = np.arange(hour_ylim[0], hour_ylim[1] + step, step)
    ax_hour.set_yticks(y_ticks)

    if 'p_value' in hour_df.columns:
        y_offset = (hour_ylim[1] - hour_ylim[0]) * 0.03
        for _, row in hour_df.iterrows():
            if row['p_value'] < sig_level:
                ax_hour.text(
                    row['hour_num'], row['coef'] + y_offset,
                    '*', ha='center', va='bottom', fontsize=10, color='black'
                )

    if shade_range is not None:
        start, end = shade_range
        ax_hour.axvspan(start, end, color='orange', alpha=0.15)

    ax_hour.spines['top'].set_visible(False)
    ax_hour.spines['right'].set_visible(False)
    ax_hour.text(
        -0.07, 1.0, 'b', transform=ax_hour.transAxes,
        fontsize=16, fontweight='bold', va='top'
    )

    # -------------------------
    # Shared y label
    # -------------------------
    fig.text(
        0.02, 0.5, 'Coefficients',
        va='center', ha='center',
        rotation='vertical',
        fontsize=18, fontweight='bold'
    )

    # -------------------------
    # Shared legend
    # -------------------------
    legend_handles = []

    legend_handles.append(
        mlines.Line2D(
            [], [], color='r', linestyle='--',
            linewidth=1.5, label='Zero effect'
        )
    )

    if show_error:
        legend_handles.append(
            mlines.Line2D([], [], color='gray', linewidth=1, label='95% CI')
        )

    legend_handles.append(
        mlines.Line2D(
            [], [], color='black', marker='*', linestyle='None',
            markersize=8, label=f'p < {sig_level}'
        )
    )

    if shade_range is not None and shade_label is not None:
        legend_handles.append(
            mpatches.Patch(color='orange', alpha=0.3, label=shade_label)
        )

    fig.legend(
        handles=legend_handles,
        fontsize=14,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    # -------------------------
    # Save / show
    # -------------------------
    plt.tight_layout(rect=[0.05, 0, 1, 1])

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')

    plt.show()
    return fig, (ax_month, ax_hour)

In [ ]:


# Usage
fig, axes = plot_month_hour_effects(
    month_df=month_effects_df,
    hour_df=hour_effects_df,
    month_ylim=(-2.5, 2.5),
    hour_ylim=(-8, 8),
    shade_range=(17, 21),
    shade_label="Peak load period",
    legend_loc='upper right',
    legend_bbox=(1.0, 1.05),
    figsize=(12, 10),
    # save_path='fig_month_hour_effects.png'
)

### with CESI 

In [ ]:
import pandas as pd
import statsmodels.api as sm

def run_voltage_temp_ols_with_CESI_clustered(
    df,
    time_col='time',
    temp_col='Temp',
    voltage_col='voltage',
    cesi_col='cesi_level',
    sensor_col='sensor_id',
    drop_first=True
):
    """
    OLS of voltage on temperature with CESI interaction,
    controlling for month and hour, with clustered SE by sensor.

    NO sensor fixed effects.
    Low CESI is the reference group.
    """

    df = df.copy()

    # Ensure datetime
    df[time_col] = pd.to_datetime(df[time_col])

    # Time controls
    df['month'] = df[time_col].dt.month
    df['hour'] = df[time_col].dt.hour

    # Drop missing
    df = df.dropna(
        subset=[temp_col, voltage_col, cesi_col, sensor_col]
    ).copy()

    # CESI indicator: High = 1, Low = 0
    df['High'] = (df[cesi_col] == 'High').astype(int)

    # Temp × CESI interaction
    df['Temp_High'] = df[temp_col] * df['High']

    # Design matrix
    cols = [temp_col, 'High', 'Temp_High', 'month', 'hour']

    X = pd.get_dummies(
        df[cols],
        columns=['month', 'hour'],
        drop_first=drop_first
    )

    X = X.astype(float)
    X = sm.add_constant(X)

    y = df[voltage_col].astype(float)

    # Clustered SE by sensor
    model = sm.OLS(y, X).fit(
        cov_type='cluster',
        cov_kwds={'groups': df[sensor_col]}
    )

    return model

In [ ]:
overall_with_cesi = run_voltage_temp_ols_with_CESI_clustered(
    hourly_volt_n_temp,
)

In [ ]:
overall_with_cesi.summary()

In [ ]:
total_temp_high = overall_with_cesi.t_test('Temp + Temp_High = 0')
print(total_temp_high.summary())

### So in plain language: a 1°C increase in temperature is associated with a −1.39V drop in voltage in High CESI areas, compared to −0.86V in Low CESI areas. 

### High CESI areas are about 60% more sensitive to temperature than Low CESI areas.

## Rolling mean windows 

## *** Workflow per sensor 

### Per sensor 

In [ ]:
import pandas as pd

def run_per_sensor_ols_summary(df, alpha=0.05):

    rows = []

    for (site_id, sensor_id), df_sensor in df.groupby(['site_id', 'sensor_id']):

        model = run_voltage_temp_ols(
            df_sensor,
            use_hac=True,
            hac_lags=24,
        )

        p_temp = model.pvalues.get('Temp', float('nan'))

        row = {
            'site_id': site_id,
            'sensor_id': sensor_id,
            'coef_Temp': model.params.get('Temp'),
            'sig_Temp': pd.notna(p_temp) and (p_temp < alpha)
        }

        # --- Month effects ---
        for k in model.params.index:
            if k.startswith('month_'):
                row[k] = model.params[k]
                row[f'{k}_sig'] = (model.pvalues[k] < alpha)

        # --- Hour effects ---
        for k in model.params.index:
            if k.startswith('hour_'):
                row[k] = model.params[k]
                row[f'{k}_sig'] = (model.pvalues[k] < alpha)

        rows.append(row)

    return pd.DataFrame(rows)

### sensor_mapping & merge with per_sensor df 

In [ ]:
sensor_mapping_copy = sensor_ea_mapping.drop(columns=['ea_code9ch', 'site_id'])

### using all dfs 

In [ ]:
per_sensor_complete = run_per_sensor_ols_summary(hourly_volt_n_temp)

per_sensor_complete = per_sensor_complete.merge(sensor_mapping_copy, on = 'sensor_id')

<div style="
    background-color:#fff3cd;
    color:#664d03;
    padding:12px;
    border-left:5px solid #ffec99;
    border-radius:6px;
    font-size:22px;
    font-weight:bold;">

1) Hour Fixed Effects

</div>

### --- Functions 

In [ ]:
import pandas as pd
import numpy as np

def build_hourly_coef_distribution(df):

    rows = []

    for h in range(1, 24):
        coef_col = f'hour_{h}'
        sig_col = f'hour_{h}_sig'

        if coef_col not in df.columns:
            continue

        # Filter to significant sensors
        df_sig = df[df[sig_col] == True]

        # Count unique sensors
        n_sensors = df_sig['sensor_id'].nunique()

        for _, row in df_sig.iterrows():
            coef = row[coef_col]

            if pd.notna(coef):
                rows.append({
                    'hour': h,
                    'coef': coef,
                    'site_id': row['site_id'],
                    'sensor_id': row['sensor_id'],
                    'cesi_level': row['cesi_level'],
                    'n_sensors': n_sensors   # <-- new column
                })

    return pd.DataFrame(rows)

In [ ]:
def plot_hourly_distribution(
    df,
    figsize=(12, 5),
    ylim=None,
    shade_range=None,
    shade_label=None,
    center_stat='mean',      # <-- 'mean' or 'median'
    show_center=True,
    show_fliers=True,        # <-- show/hide outliers
    legend_loc='best',
    legend_bbox=None
):
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    import matplotlib.patches as mpatches
    import matplotlib.lines as mlines

    plt.figure(figsize=figsize)

    # --- Boxplot ---
    sns.boxplot(
        x='hour',
        y='coef',
        data=df,
        showfliers=show_fliers   # <-- control outliers
    )

    legend_handles = []

    # --- Shading ---
    if shade_range is not None:
        start, end = shade_range

        plt.axvspan(start - 1, end - 1, color='orange', alpha=0.15)

        if shade_label is not None:
            legend_handles.append(
                mpatches.Patch(
                    color='orange',
                    alpha=0.15,
                    label=shade_label
                )
            )

    # --- Center line (mean or median) ---
    if show_center:

        if center_stat not in ['mean', 'median']:
            raise ValueError("center_stat must be 'mean' or 'median'")

        if center_stat == 'mean':
            center_df = (
                df.groupby('hour', as_index=False)['coef']
                  .mean()
                  .sort_values('hour')
            )
        else:
            center_df = (
                df.groupby('hour', as_index=False)['coef']
                  .median()
                  .sort_values('hour')
            )

        plt.plot(
            center_df['hour'] - 1,
            center_df['coef'],
            marker='o',
            linewidth=2,
            color='black'
        )

        legend_handles.append(
            mlines.Line2D(
                [],
                [],
                color='black',
                marker='o',
                linewidth=2,
                label=f'{center_stat.capitalize()} coefficient'
            )
        )

    # --- Labels ---
    plt.xlabel("Hour of day", fontsize=16, labelpad=15, fontweight='bold')
    plt.ylabel("Average Change in Voltage", fontsize=16, labelpad=15, fontweight='bold')

    # Reference line
    plt.axhline(0, linestyle='--', color='r')

    # Ticks
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # --- Y limits ---
    if ylim is not None:
        plt.ylim(ylim)
    else:
        max_val = np.ceil(np.max(np.abs(df['coef'])))
        plt.ylim(-max_val, max_val)

    # --- Legend ---
    if legend_handles:
        plt.legend(
            handles=legend_handles,
            fontsize=14,
            frameon=True,
            loc=legend_loc,
            bbox_to_anchor=legend_bbox
        )

    # Remove top/right spines
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_hourly_distribution_by_cesi(
    df,
    figsize=(12, 5),
    ylim=None,
    shade_range=None,
    shade_label=None,
    show_mean=True,
    show_fliers=True,          # <-- NEW FLAG
    legend_loc='best',
    legend_bbox=None,
    cesi_label_map=None
):
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    import matplotlib.patches as mpatches
    import matplotlib.lines as mlines

    plt.figure(figsize=figsize)

    # Boxplot
    sns.boxplot(
        x='hour',
        y='coef',
        hue='cesi_level',
        data=df,
        showfliers=show_fliers   # <-- APPLY HERE
    )

    legend_handles = []

    # Shading
    if shade_range is not None:
        start, end = shade_range
        plt.axvspan(start - 1, end - 1, color='orange', alpha=0.15)

        if shade_label is not None:
            legend_handles.append(
                mpatches.Patch(
                    color='orange',
                    alpha=0.15,
                    label=shade_label
                )
            )

    # Mean lines
    if show_mean:
        cesi_levels = df['cesi_level'].dropna().unique()

        for level in cesi_levels:
            mean_df = (
                df[df['cesi_level'] == level]
                .groupby('hour', as_index=False)['coef']
                .mean()
                .sort_values('hour')
            )

            label = level
            if cesi_label_map is not None:
                label = cesi_label_map.get(level, level)

            plt.plot(
                mean_df['hour'] - 1,
                mean_df['coef'],
                marker='o',
                linewidth=2,
                label=f'{label} mean'
            )

    # Labels
    plt.xlabel("Hour of day", fontsize=16, labelpad=15, fontweight='bold')
    plt.ylabel("Average Change in Voltage", fontsize=16, labelpad=15, fontweight='bold')

    plt.axhline(0, linestyle='--', color='r')

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    if ylim is not None:
        plt.ylim(ylim)
    else:
        max_val = np.ceil(np.max(np.abs(df['coef'])))
        plt.ylim(-max_val, max_val)

    # --- Legend cleanup ---
    ax = plt.gca()
    handles, labels = ax.get_legend_handles_labels()

    if cesi_label_map is not None:
        new_labels = []
        for lab in labels:
            base = lab.replace(" mean", "")
            mapped = cesi_label_map.get(base, base)
            if "mean" in lab:
                new_labels.append(f"{mapped} mean")
            else:
                new_labels.append(mapped)
        labels = new_labels

    # Add shading handles first
    if legend_handles:
        handles = legend_handles + handles
        labels = [h.get_label() for h in legend_handles] + labels

    plt.legend(
        handles=handles,
        labels=labels,
        fontsize=14,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    # Remove spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
def compare_cesi_hourly_coefficients(
    df,
    baseline_level='Low',
    comparison_level='High',
    alpha=0.05
):
    import pandas as pd
    import numpy as np

    from scipy.stats import mannwhitneyu

    results = []

    # -------------------
    # Loop through hours
    # -------------------
    for hour in sorted(df['hour'].dropna().unique()):

        hour_df = df[df['hour'] == hour]

        baseline_vals = (
            hour_df[hour_df['cesi_level'] == baseline_level]['coef']
            .dropna()
        )

        comparison_vals = (
            hour_df[hour_df['cesi_level'] == comparison_level]['coef']
            .dropna()
        )

        # Skip if insufficient data
        if len(baseline_vals) < 2 or len(comparison_vals) < 2:
            continue

        # -------------------
        # Mann-Whitney U test
        # -------------------
        stat, p_value = mannwhitneyu(
            comparison_vals,
            baseline_vals,
            alternative='two-sided'
        )

        # -------------------
        # Medians
        # -------------------
        baseline_median = baseline_vals.median()
        comparison_median = comparison_vals.median()

        delta = comparison_median - baseline_median

        results.append({
            'hour': hour,

            f'n_{baseline_level}': len(baseline_vals),
            f'n_{comparison_level}': len(comparison_vals),

            f'median_{baseline_level}': baseline_median,
            f'median_{comparison_level}': comparison_median,

            'median_delta': delta,

            'u_statistic': stat,
            'p_value': p_value,

            'significant': p_value < alpha
        })

    # -------------------
    # Convert to dataframe
    # -------------------
    results_df = pd.DataFrame(results)

    # -------------------
    # Significance stars
    # -------------------
    def significance_stars(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return ''

    results_df['sig_stars'] = (
        results_df['p_value']
        .apply(significance_stars)
    )

    return results_df

In [ ]:
def plot_hourly_median_delta_by_cesi(
    df,
    test_df=None,
    baseline_level='High',
    comparison_level='Low',
    figsize=(12, 5),
    ylim=None,
    shade_range=None,
    shade_label=None,
    color='firebrick',
    marker='o',
    linewidth=3,
    linestyle='-',
    show_grid_lines=True,
    annotate=False,
    annotate_sig=True,
    sig_y_offset=0.25,
    sig_fontsize=16,
    show_sig_legend=True,
    legend_loc='best',
    legend_bbox=None,
    ylabel='Median Coefficient Delta'
):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import matplotlib.lines as mlines

    df = df.copy()

    # -------------------
    # Hourly medians
    # -------------------
    median_df = (
        df.groupby(['hour', 'cesi_level'])['coef']
          .median()
          .reset_index()
    )

    # -------------------
    # Pivot to CESI columns
    # -------------------
    pivot_df = median_df.pivot(
        index='hour',
        columns='cesi_level',
        values='coef'
    )

    # -------------------
    # Delta
    # Example default: Low - High
    # -------------------
    pivot_df['delta'] = (
        pivot_df[comparison_level] -
        pivot_df[baseline_level]
    )

    pivot_df = pivot_df.reset_index().sort_values('hour')

    # -------------------
    # Plot
    # -------------------
    plt.figure(figsize=figsize)

    legend_handles = []

    # -------------------
    # Shading
    # -------------------
    if shade_range is not None:
        start, end = shade_range

        plt.axvspan(
            start,
            end,
            color='orange',
            alpha=0.15
        )

        if shade_label is not None:
            legend_handles.append(
                mpatches.Patch(
                    color='orange',
                    alpha=0.15,
                    label=shade_label
                )
            )

    # -------------------
    # Delta line
    # -------------------
    plt.plot(
        pivot_df['hour'],
        pivot_df['delta'],
        color=color,
        marker=marker,
        linewidth=linewidth,
        linestyle=linestyle,
        label=f'{comparison_level} - {baseline_level}'
    )

    # -------------------
    # Zero line
    # -------------------
    plt.axhline(
        0,
        linestyle='--',
        color='black',
        linewidth=1.5,
        label='No Difference'
    )

    # -------------------
    # Optional numeric annotations
    # -------------------
    if annotate:
        for _, row in pivot_df.iterrows():
            plt.text(
                row['hour'],
                row['delta'],
                f"{row['delta']:.2f}",
                fontsize=9,
                ha='center',
                va='bottom'
            )

    # -------------------
    # Significance stars
    # -------------------
    if test_df is not None and annotate_sig:

        sig_df = test_df[
            (test_df['significant'] == True) &
            (test_df['sig_stars'].notna()) &
            (test_df['sig_stars'] != '')
        ]

        sig_map = dict(zip(sig_df['hour'], sig_df['sig_stars']))

        for _, row in pivot_df.iterrows():

            hour = row['hour']

            if hour in sig_map:

                y = row['delta']

                va = 'bottom' if y >= 0 else 'top'
                offset = sig_y_offset if y >= 0 else -sig_y_offset

                plt.text(
                    hour,
                    y + offset,
                    sig_map[hour],
                    fontsize=sig_fontsize,
                    fontweight='bold',
                    ha='center',
                    va=va
                )

    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Hour of day",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    plt.ylabel(
        ylabel,
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    # -------------------
    # Limits
    # -------------------
    if ylim is not None:
        plt.ylim(ylim)
    else:
        max_val = np.ceil(np.max(np.abs(pivot_df['delta'])))
        plt.ylim(-max_val, max_val)

    # -------------------
    # Gridlines
    # -------------------
    if show_grid_lines:
        plt.grid(axis='y', linestyle='--', alpha=0.3)

    # -------------------
    # Ticks
    # -------------------
    plt.xticks(range(0, 24), fontsize=12)
    plt.yticks(fontsize=12)

    # -------------------
    # Legend
    # -------------------
    ax = plt.gca()

    handles, labels = ax.get_legend_handles_labels()

    if legend_handles:
        handles = legend_handles + handles
        labels = [h.get_label() for h in legend_handles] + labels

    # -------------------
    # Significance legend
    # -------------------
    if show_sig_legend:

        sig_1 = mlines.Line2D(
            [], [],
            color='black',
            marker='$*$',
            linestyle='None',
            markersize=6,
            label='p < 0.05'
        )

        sig_2 = mlines.Line2D(
            [], [],
            color='black',
            marker='$**$',
            linestyle='None',
            markersize=14,
            label='p < 0.01'
        )

        handles.extend([sig_1, sig_2])

        labels.extend([
            'p < 0.05',
            'p < 0.01',
        ])

    plt.legend(
        handles=handles,
        labels=labels,
        fontsize=14,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

    return pivot_df

In [ ]:
def get_hourly_sensor_counts(df):
    import pandas as pd

    # -------------------
    # Unique sensor counts
    # -------------------
    counts_df = (
        df.groupby(['hour', 'cesi_level'])['sensor_id']
          .nunique()
          .reset_index(name='n_sensors')
    )

    # -------------------
    # Pivot to wide format
    # -------------------
    counts_pivot = counts_df.pivot(
        index='hour',
        columns='cesi_level',
        values='n_sensors'
    ).fillna(0)

    # -------------------
    # Ensure columns exist
    # -------------------
    for col in ['Low', 'High']:
        if col not in counts_pivot.columns:
            counts_pivot[col] = 0

    # -------------------
    # Convert to int
    # -------------------
    counts_pivot['Low'] = counts_pivot['Low'].astype(int)
    counts_pivot['High'] = counts_pivot['High'].astype(int)

    # -------------------
    # Total sensors
    # -------------------
    counts_pivot['Total'] = (
        counts_pivot['Low'] +
        counts_pivot['High']
    )

    # -------------------
    # Reset index
    # -------------------
    counts_pivot = counts_pivot.reset_index()

    # Reorder columns
    counts_pivot = counts_pivot[
        ['hour', 'Low', 'High', 'Total']
    ]

    return counts_pivot

In [ ]:
def plot_hourly_distribution_bars(
    df,
    figsize=(12, 5),
    ylim=None,
    shade_range=None,
    shade_label=None,
    center_stat='mean',      # <-- 'mean' or 'median'
    show_center=True,
    show_n=True,
    show_fliers=True,        # <-- NEW
    legend_loc='best',
    legend_bbox=None
):
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    import matplotlib.patches as mpatches
    import matplotlib.lines as mlines
    from matplotlib.ticker import PercentFormatter

    fig, ax1 = plt.subplots(figsize=figsize)

    legend_handles = []

    # --- Secondary axis FIRST (bars behind) ---
    if show_n and 'n_sensors' in df.columns:

        ax2 = ax1.twinx()

        n_df = (
            df[['hour', 'n_sensors']]
            .drop_duplicates()
            .sort_values('hour')
            .copy()
        )

        total_sensors = df['sensor_id'].nunique()
        n_df['pct_sensors'] = (n_df['n_sensors'] / total_sensors) * 100

        ax2.bar(
            n_df['hour'] - 1,
            n_df['pct_sensors'],
            width=0.4,
            alpha=0.2,
            color='green',
            zorder=0
        )

        ax2.set_ylabel("% of sensors", fontsize=16, labelpad=15, fontweight='bold')
        ax2.tick_params(axis='y', labelsize=12)
        ax2.set_ylim(0, 100)
        ax2.yaxis.set_major_formatter(PercentFormatter())

        ax2.spines['top'].set_visible(False)

        legend_handles.append(
            mlines.Line2D(
                [], [],
                color='green',
                linewidth=8,
                alpha=0.2,
                label='% of sensors'
            )
        )

    # --- Distribution ---
    sns.boxplot(
        x='hour',
        y='coef',
        data=df,
        ax=ax1,
        showfliers=show_fliers,   # <-- HERE
        zorder=1
    )

    # --- Shading ---
    if shade_range is not None:
        start, end = shade_range
        ax1.axvspan(start - 1, end - 1, color='orange', alpha=0.15)

        if shade_label is not None:
            legend_handles.insert(
                0,
                mpatches.Patch(
                    color='orange',
                    alpha=0.15,
                    label=shade_label
                )
            )

    # --- Center line ---
    if show_center:

        if center_stat not in ['mean', 'median']:
            raise ValueError("center_stat must be 'mean' or 'median'")

        if center_stat == 'mean':
            center_df = (
                df.groupby('hour', as_index=False)['coef']
                  .mean()
                  .sort_values('hour')
            )
        else:
            center_df = (
                df.groupby('hour', as_index=False)['coef']
                  .median()
                  .sort_values('hour')
            )

        ax1.plot(
            center_df['hour'] - 1,
            center_df['coef'],
            marker='o',
            linewidth=2,
            color='black',
            zorder=2
        )

        legend_handles.append(
            mlines.Line2D(
                [], [],
                color='black',
                marker='o',
                linewidth=2,
                label=f'{center_stat.capitalize()} coefficient'
            )
        )

    # --- Primary axis ---
    ax1.set_xlabel("Hour of day", fontsize=16, labelpad=15, fontweight='bold')
    ax1.set_ylabel("Coefficients", fontsize=16, labelpad=15, fontweight='bold')

    ax1.axhline(0, linestyle='--', color='r')

    ax1.tick_params(axis='x', labelsize=12)
    ax1.tick_params(axis='y', labelsize=12)

    if ylim is not None:
        ax1.set_ylim(ylim)
    else:
        max_val = np.ceil(np.max(np.abs(df['coef'])))
        ax1.set_ylim(-max_val, max_val)

    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # --- Legend ---
    if legend_handles:
        ax1.legend(
            handles=legend_handles,
            fontsize=14,
            frameon=True,
            loc=legend_loc,
            bbox_to_anchor=legend_bbox
        )

    plt.tight_layout()
    plt.show()

### distribution plot  

In [ ]:
distribution_hour_df = build_hourly_coef_distribution(per_sensor_complete)

### Hourly 

In [ ]:
plot_hourly_distribution(
    distribution_hour_df,
    ylim=(-25, 20),
    shade_range=(17, 21),
    shade_label="Peak load period",
    legend_loc='upper left',
    legend_bbox=(0.9, 1), 
    # center_stat='median',
    show_fliers=False,
)

### Hourly - by CESI  

In [ ]:
plot_hourly_distribution_by_cesi(
    distribution_hour_df,
    figsize=(14, 6),
    ylim=(-25, 20),
    shade_range=(17, 20),
    shade_label="Peak load period",
    legend_loc='upper left',
    legend_bbox=(0.85, 1), 
    cesi_label_map={
        'Low': 'Low CESI',
        'High': 'High CESI'
    },
    show_mean=False, 
    show_fliers=False,
)

### Check for stat. significant difference 

In [ ]:
hourly_test_df = compare_cesi_hourly_coefficients(
    distribution_hour_df,
    baseline_level='High',
    comparison_level='Low'
)

In [ ]:
hourly_test_df

### Delta Median (Low - High CESI) plot 

In [ ]:
hourly_delta_df = plot_hourly_median_delta_by_cesi(
    distribution_hour_df,
    test_df=hourly_test_df,
    baseline_level='High',
    comparison_level='Low',
    figsize=(14, 6),
    ylim=(-5, 5),
    shade_range=(17, 21),
    shade_label="Peak load period",
    legend_loc='upper left',
    legend_bbox=(0.85, 1.05),
    ylabel='Delta Median: Low – High CESI'
)

### `positive values`: Means the Low CESI areas had higher voltages / High CESI had lower voltages 

### `negative values`: Means the Low CESI areas had lower voltages / High CESI had higher voltages 

#### in evening/night, High CESI areas generally have poorer voltages 

#### maybe longer LV lines, higher voltage drops, potentially poor joints/{even illegal connections} which could lead to high Resistance points, hence more drops during peak load periods? 

#### in the morning, low CESI areas generally have poorer voltages 

#### maybe use of irons, kettles, microwaves, etc when prepping for work & school (but why doesn't the same trend continue in the evening?) 

#### I expected Low CESI areas to have lower voltages at night though (cos I assumed they would have more cooling equipment and appliances in general) 

### hourly sensor stats 

In [ ]:
hourly_counts_df = get_hourly_sensor_counts(distribution_hour_df)

In [ ]:
# hourly_counts_df

### showing % reporting sensors in addition 

In [ ]:
# plot_hourly_distribution(
#     distribution_hour_df,
#     figsize = (13,6), 
#     ylim=(-25, 20),
#     shade_range=(17, 20),
#     shade_label="Peak load period",
#     legend_loc='upper left',
#     legend_bbox=(0.45, 1), 
#     show_mean=False, 
#     show_fliers=False
# )

In [ ]:
# plot_hourly_distribution_bars(
#     distribution_hour_df,
#     figsize = (13,6), 
#     ylim=(-20, 20),
#     # shade_range=(17, 20),
#     # shade_label="Peak load period",
#     center_stat='median', 
#     legend_loc='upper left',
#     legend_bbox=(0.60, 1.20), 
#     show_fliers=False
# )

### Among sensors where the effect is statistically significant, what is the average (distribution of) hourly effect? 

<div style="
    background-color:#f2f39e;
    color:#664d03;
    padding:12px;
    border-left:5px solid #ffec99;
    border-radius:6px;
    font-size:22px;
    font-weight:bold;">

2) Month Fixed Effects

</div>

### --- Functions 

In [ ]:
import pandas as pd
import numpy as np

def build_monthly_coef_distribution(df):

    rows = []

    for m in range(2, 13):  # month_2 to month_12; January is baseline
        coef_col = f'month_{m}'
        sig_col = f'month_{m}_sig'

        if coef_col not in df.columns:
            continue

        # Filter to significant sensors
        df_sig = df[df[sig_col] == True]

        # Count unique significant sensors for this month
        n_sensors = df_sig['sensor_id'].nunique()

        for _, row in df_sig.iterrows():
            coef = row[coef_col]

            if pd.notna(coef):
                rows.append({
                    'month': m,
                    'coef': coef,
                    'site_id': row['site_id'],
                    'sensor_id': row['sensor_id'],
                    'cesi_level': row['cesi_level'],
                    'n_sensors': n_sensors
                })

    return pd.DataFrame(rows)

In [ ]:
def plot_monthly_distribution(
    df,
    figsize=(12, 5),
    ylim=None,
    center_stat='mean',   # 'mean', 'median', or None
    show_fliers=True,     # <-- NEW
    legend_loc='best',
    legend_bbox=None
):
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    import calendar
    import matplotlib.lines as mlines

    plt.figure(figsize=figsize)

    # Month labels
    df = df.copy()
    df['month_label'] = df['month'].apply(lambda m: calendar.month_abbr[m])

    order = [calendar.month_abbr[m] for m in range(2, 13)]

    # --- Boxplot ---
    sns.boxplot(
        x='month_label',
        y='coef',
        data=df,
        order=order,
        showfliers=show_fliers   # <-- APPLY HERE
    )

    legend_handles = []

    # --- Overlay center statistic ---
    if center_stat is not None:

        if center_stat not in ['mean', 'median']:
            raise ValueError("center_stat must be 'mean', 'median', or None")

        if center_stat == 'mean':
            center_df = (
                df.groupby('month', as_index=False)['coef']
                  .mean()
                  .sort_values('month')
            )
            label = 'Mean coefficient'
        else:
            center_df = (
                df.groupby('month', as_index=False)['coef']
                  .median()
                  .sort_values('month')
            )
            label = 'Median coefficient'

        # map months (Feb=0,...,Dec=10)
        plt.plot(
            center_df['month'] - 2,
            center_df['coef'],
            marker='o',
            linewidth=2,
            color='black'
        )

        legend_handles.append(
            mlines.Line2D(
                [], [],
                color='black',
                marker='o',
                linewidth=2,
                label=label
            )
        )

    # Axis labels
    plt.xlabel("Month", fontsize=16, labelpad=15, fontweight='bold')
    plt.ylabel("Average Change in Voltage", fontsize=16, labelpad=15, fontweight='bold')

    # Reference line
    plt.axhline(0, linestyle='--', color='r')

    # Ticks
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # Y limits
    if ylim is not None:
        plt.ylim(ylim)
    else:
        max_val = np.ceil(np.max(np.abs(df['coef'])))
        plt.ylim(-max_val, max_val)

    # Legend
    if legend_handles:
        plt.legend(
            handles=legend_handles,
            fontsize=14,
            frameon=True,
            loc=legend_loc,
            bbox_to_anchor=legend_bbox
        )

    # Remove spines
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_monthly_distribution_by_cesi(
    df,
    figsize=(12, 5),
    ylim=None,
    show_center=True,
    center_stat='mean',
    center_style='-',
    show_fliers=True,   # <-- NEW
    legend_loc='best',
    legend_bbox=None,
    cesi_label_map=None
):
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    import calendar

    plt.figure(figsize=figsize)

    df = df.copy()
    df['month_label'] = df['month'].apply(lambda m: calendar.month_abbr[m])

    month_order = [calendar.month_abbr[m] for m in range(2, 13)]

    sns.boxplot(
        x='month_label',
        y='coef',
        hue='cesi_level',
        data=df,
        order=month_order,
        showfliers=show_fliers
    )

    if show_center:
        if center_stat not in ['mean', 'median']:
            raise ValueError("center_stat must be 'mean' or 'median'")

        for level in df['cesi_level'].dropna().unique():
            subset = df[df['cesi_level'] == level]

            if center_stat == 'mean':
                center_df = (
                    subset.groupby('month', as_index=False)['coef']
                          .mean()
                          .sort_values('month')
                )
            else:
                center_df = (
                    subset.groupby('month', as_index=False)['coef']
                          .median()
                          .sort_values('month')
                )

            label_base = cesi_label_map.get(level, level) if cesi_label_map else level

            plt.plot(
                center_df['month'] - 2,
                center_df['coef'],
                linestyle=center_style,
                marker='o',
                linewidth=2,
                label=f'{label_base} {center_stat}'
            )

    plt.xlabel("Month", fontsize=16, labelpad=15, fontweight='bold')
    plt.ylabel("Average Change in Voltage", fontsize=16, labelpad=15, fontweight='bold')
    plt.axhline(0, linestyle='--', color='r')

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    if ylim is not None:
        plt.ylim(ylim)
    else:
        max_val = np.ceil(np.max(np.abs(df['coef'])))
        plt.ylim(-max_val, max_val)

    ax = plt.gca()
    handles, labels = ax.get_legend_handles_labels()

    if cesi_label_map is not None:
        new_labels = []
        for lab in labels:
            parts = lab.split()
            base = parts[0]
            mapped = cesi_label_map.get(base, base)

            if len(parts) > 1:
                new_labels.append(f"{mapped} {' '.join(parts[1:])}")
            else:
                new_labels.append(mapped)

        labels = new_labels

    plt.legend(
        handles=handles,
        labels=labels,
        fontsize=14,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
def get_monthly_sensor_counts(df):
    import pandas as pd
    import calendar

    # -------------------
    # Unique sensor counts
    # -------------------
    counts_df = (
        df.groupby(['month', 'cesi_level'])['sensor_id']
          .nunique()
          .reset_index(name='n_sensors')
    )

    # -------------------
    # Pivot to wide format
    # -------------------
    counts_pivot = counts_df.pivot(
        index='month',
        columns='cesi_level',
        values='n_sensors'
    ).fillna(0)

    # -------------------
    # Ensure columns exist
    # -------------------
    for col in ['Low', 'High']:
        if col not in counts_pivot.columns:
            counts_pivot[col] = 0

    # -------------------
    # Convert to int
    # -------------------
    counts_pivot['Low'] = counts_pivot['Low'].astype(int)
    counts_pivot['High'] = counts_pivot['High'].astype(int)

    # -------------------
    # Total
    # -------------------
    counts_pivot['Total'] = (
        counts_pivot['Low'] +
        counts_pivot['High']
    )

    # -------------------
    # Month labels
    # -------------------
    counts_pivot = counts_pivot.reset_index()

    counts_pivot['month_label'] = counts_pivot['month'].apply(
        lambda m: calendar.month_abbr[m]
    )

    # Reorder columns
    counts_pivot = counts_pivot[
        ['month', 'month_label', 'Low', 'High', 'Total']
    ]

    return counts_pivot

In [ ]:
def compare_cesi_coefficients_by_month(
    df,
    baseline_level='Low',
    comparison_level='High',
    alpha=0.05
):
    import pandas as pd
    import numpy as np

    from scipy.stats import mannwhitneyu

    results = []

    # -------------------
    # Loop through months
    # -------------------
    for month in sorted(df['month'].dropna().unique()):

        month_df = df[df['month'] == month]

        baseline_vals = (
            month_df[month_df['cesi_level'] == baseline_level]['coef']
            .dropna()
        )

        comparison_vals = (
            month_df[month_df['cesi_level'] == comparison_level]['coef']
            .dropna()
        )

        # Skip if insufficient data
        if len(baseline_vals) < 2 or len(comparison_vals) < 2:
            continue

        # -------------------
        # Mann-Whitney U test
        # -------------------
        stat, p_value = mannwhitneyu(
            comparison_vals,
            baseline_vals,
            alternative='two-sided'
        )

        # -------------------
        # Medians
        # -------------------
        baseline_median = baseline_vals.median()
        comparison_median = comparison_vals.median()

        delta = comparison_median - baseline_median

        results.append({
            'month': month,

            f'n_{baseline_level}': len(baseline_vals),
            f'n_{comparison_level}': len(comparison_vals),

            f'median_{baseline_level}': baseline_median,
            f'median_{comparison_level}': comparison_median,

            'median_delta': delta,

            'u_statistic': stat,
            'p_value': p_value,

            'significant': p_value < alpha
        })

    # -------------------
    # Convert to dataframe
    # -------------------
    results_df = pd.DataFrame(results)

    # -------------------
    # Significance stars
    # -------------------
    def significance_stars(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return ''

    results_df['sig_stars'] = (
        results_df['p_value']
        .apply(significance_stars)
    )

    return results_df

In [ ]:
def plot_monthly_median_delta(
    df,
    test_df=None,
    baseline_level='Low',
    comparison_level='High',
    figsize=(10, 5),
    color='firebrick',
    marker='o',
    linewidth=3,
    linestyle='-',
    ylim=None,
    show_grid_lines=True,
    annotate=True,
    annotate_sig=True,
    sig_y_offset=0.25,
    sig_fontsize=16,
    show_sig_legend=True,
    legend_loc='best',
    legend_bbox=None,
    ylabel='Delta Median Coefficient',
    title=None
):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import calendar
    import matplotlib.lines as mlines

    df = df.copy()

    median_df = (
        df.groupby(['month', 'cesi_level'])['coef']
          .median()
          .reset_index()
    )

    pivot_df = median_df.pivot(
        index='month',
        columns='cesi_level',
        values='coef'
    )

    pivot_df['delta'] = (
        pivot_df[comparison_level] -
        pivot_df[baseline_level]
    )

    pivot_df = pivot_df.reset_index()

    pivot_df['month_label'] = pivot_df['month'].apply(
        lambda m: calendar.month_abbr[m]
    )

    plt.figure(figsize=figsize)

    plt.plot(
        pivot_df['month_label'],
        pivot_df['delta'],
        color=color,
        marker=marker,
        linewidth=linewidth,
        linestyle=linestyle,
        label=f'{comparison_level} - {baseline_level}'
    )

    plt.axhline(
        0,
        linestyle='--',
        color='black',
        linewidth=1.5,
        label='No Difference'
    )

    if annotate:
        for _, row in pivot_df.iterrows():
            plt.text(
                row['month_label'],
                row['delta'],
                f"{row['delta']:.2f}",
                fontsize=10,
                ha='center',
                va='bottom'
            )

    # -------------------
    # Significance stars
    # -------------------
    if test_df is not None and annotate_sig:
        sig_df = test_df[
            (test_df['significant'] == True) &
            (test_df['sig_stars'].notna()) &
            (test_df['sig_stars'] != '')
        ].copy()

        sig_map = dict(zip(sig_df['month'], sig_df['sig_stars']))

        for _, row in pivot_df.iterrows():
            month = row['month']

            if month in sig_map:
                y = row['delta']

                va = 'bottom' if y >= 0 else 'top'
                offset = sig_y_offset if y >= 0 else -sig_y_offset

                plt.text(
                    row['month_label'],
                    y + offset,
                    sig_map[month],
                    fontsize=sig_fontsize,
                    fontweight='bold',
                    ha='center',
                    va=va
                )

    plt.xlabel(
        "Month",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    plt.ylabel(
        ylabel,
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    if title is not None:
        plt.title(title)

    if ylim is not None:
        plt.ylim(ylim)

    if show_grid_lines:
        plt.grid(axis='y', linestyle='--', alpha=0.3)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    ax = plt.gca()

    handles, labels = ax.get_legend_handles_labels()

    # -------------------
    # Significance legend
    # -------------------
    if show_sig_legend:
        sig_1 = mlines.Line2D(
            [], [],
            color='black',
            marker='$*$',
            linestyle='None',
            markersize=6,
            label='p < 0.05'
        )

        handles.extend([sig_1])
        labels.extend([
            'p < 0.05',
        ])

    plt.legend(
        handles=handles,
        labels=labels,
        fontsize=12,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

    return pivot_df

In [ ]:
def plot_temp_coef_histogram(
    df,
    bins=20,
    figsize=(8, 5),
    show_kde=True,
    normalize=False,
    show_center=True,
    show_grid_lines=True,

    # --- Zero reference line ---
    zero_line_value=0,
    zero_line_label='Zero Coefficient',
    zero_line_color='red',
    zero_line_style='--',

    # --- Overall coefficient line ---
    overall_temp_coeff=None,
    overall_temp_coeff_label='Overall Temp Coeff',
    overall_temp_coeff_color='darkorange',
    overall_temp_coeff_style='--',

    # --- Legend ---
    legend_loc='best',
    legend_bbox=None
):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.ticker import PercentFormatter

    plt.figure(figsize=figsize)

    # -------------------
    # Histogram
    # -------------------
    if normalize:
        sns.histplot(
            df['coef_Temp'],
            bins=bins,
            kde=show_kde,
            stat='percent', 
            color = 'grey'
        )
    else:
        sns.histplot(
            df['coef_Temp'],
            bins=bins,
            kde=show_kde,
            stat='count', 
            color = 'grey'
        )

    # -------------------
    # Zero reference line
    # -------------------
    plt.axvline(
        zero_line_value,
        linestyle=zero_line_style,
        color=zero_line_color,
        linewidth=2,
        label=zero_line_label
    )

    # -------------------
    # Mean / Median
    # -------------------
    if show_center:
        mean_val = df['coef_Temp'].mean()
        median_val = df['coef_Temp'].median()

        plt.axvline(
            mean_val,
            color='black',
            linestyle='-',
            linewidth=2,
            label='Mean'
        )

        plt.axvline(
            median_val,
            color='black',
            linestyle='--',
            linewidth=2,
            label='Median'
        )

    # -------------------
    # Overall temp coefficient line
    # -------------------
    if overall_temp_coeff is not None:
        plt.axvline(
            overall_temp_coeff,
            color=overall_temp_coeff_color,
            linestyle=overall_temp_coeff_style,
            linewidth=2,
            label=overall_temp_coeff_label
        )

    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Change in Voltage",
        fontsize=14,
        labelpad=15,
        fontweight='bold'
    )

    ylabel = "Percentage of Sensors (%)" if normalize else "Count"

    plt.ylabel(
        ylabel,
        fontsize=14,
        labelpad=15,
        fontweight='bold'
    )

    # -------------------
    # Percent formatting
    # -------------------
    ax = plt.gca()

    if normalize:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))

    # -------------------
    # Gridlines
    # -------------------
    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    # -------------------
    # Ticks
    # -------------------
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # -------------------
    # Legend
    # -------------------
    plt.legend(
        frameon=False,
        fontsize=11,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_temp_coef_histogram_by_cesi(
    df,
    bins=20,
    figsize=(8, 5),
    normalize=True,
    show_center=True,
    show_kde=True,
    show_grid_lines=True,
    fill_value=True,
    cesi_label_map=None,
    legend_loc='best',
    legend_bbox=None,
    zero_line_label='No Temperature Effect'
):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.ticker import PercentFormatter
    import matplotlib.lines as mlines

    plt.figure(figsize=figsize)
    ax = plt.gca()

    stat = 'percent' if normalize else 'count'

    # -------------------
    # Consistent palette
    # -------------------
    levels = df['cesi_level'].dropna().unique()

    palette = dict(
        zip(levels, sns.color_palette(n_colors=len(levels)))
    )

    # -------------------
    # Histogram + KDE
    # -------------------
    sns.histplot(
        data=df,
        x='coef_Temp',
        hue='cesi_level',
        bins=bins,
        stat=stat,
        element='step',
        fill=fill_value,
        alpha=0.2,
        kde=show_kde,
        common_norm=False,
        palette=palette,
        ax=ax
    )

    # -------------------
    # Reference line
    # -------------------
    plt.axvline(
        0,
        linestyle='--',
        color='r'
    )

    # -------------------
    # Median lines
    # -------------------
    center_handles = []

    if show_center:

        for level in levels:

            sub = df[df['cesi_level'] == level]

            median_val = sub['coef_Temp'].median()

            label = (
                cesi_label_map.get(level, level)
                if cesi_label_map else level
            )

            plt.axvline(
                median_val,
                linestyle='--',
                linewidth=2,
                color=palette[level],
                alpha=0.9
            )

            center_handles.append(
                mlines.Line2D(
                    [],
                    [],
                    color=palette[level],
                    linestyle='--',
                    linewidth=2,
                    label=f'{label}'
                )
            )

    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Change in Voltage",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    ylabel = (
        "Percentage of Sensors (%)"
        if normalize else "Count"
    )

    plt.ylabel(
        ylabel,
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    # -------------------
    # Percent formatting
    # -------------------
    if normalize:
        ax.yaxis.set_major_formatter(
            PercentFormatter(xmax=100)
        )

    # -------------------
    # Gridlines
    # -------------------
    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    # -------------------
    # Legend
    # -------------------
    handles, labels = ax.get_legend_handles_labels()

    clean_handles = []
    clean_labels = []

    for h, l in zip(handles, labels):

        if l in levels:

            clean_handles.append(h)
            clean_labels.append(l)

    # Apply CESI label map
    if cesi_label_map is not None:
        clean_labels = [
            cesi_label_map.get(l, l)
            for l in clean_labels
        ]

    # Add center handles
    clean_handles += center_handles
    clean_labels += [
        h.get_label()
        for h in center_handles
    ]

    # Add zero-effect line to legend
    zero_handle = mlines.Line2D(
        [],
        [],
        color='red',
        linestyle='--',
        linewidth=2,
        label=zero_line_label
    )

    clean_handles.append(zero_handle)
    clean_labels.append(zero_line_label)

    if len(clean_handles) > 0:

        plt.legend(
            handles=clean_handles,
            labels=clean_labels,
            fontsize=14,
            frameon=True,
            loc=legend_loc,
            bbox_to_anchor=legend_bbox
        )

    # -------------------
    # Ticks
    # -------------------
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
### revised function - cmap for CESI 

In [ ]:
def plot_temp_coef_histogram_by_cesi(
    df,
    bins=20,
    figsize=(8, 5),
    normalize=True,
    show_center=True,
    show_kde=True,
    show_grid_lines=True,
    fill_value=True,
    cesi_label_map=None,
    legend_loc='best',
    legend_bbox=None,
    zero_line_label='No Temperature Effect'
):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.ticker import PercentFormatter
    import matplotlib.lines as mlines

    plt.figure(figsize=figsize)
    ax = plt.gca()

    stat = 'percent' if normalize else 'count'

    # -------------------
    # Consistent palette
    # -------------------
    levels = df['cesi_level'].dropna().unique()

    cesi_colors = {
        'Low':    '#6a51a3',
        'Medium': '#238b45',
        'High':   '#8c2d04'
    }

    palette = {
        level: cesi_colors.get(level, fallback)
        for level, fallback in zip(
            levels,
            sns.color_palette(n_colors=len(levels)).as_hex()
        )
    }

    # -------------------
    # Histogram + KDE
    # -------------------
    sns.histplot(
        data=df,
        x='coef_Temp',
        hue='cesi_level',
        bins=bins,
        stat=stat,
        element='step',
        fill=fill_value,
        alpha=0.2,
        kde=show_kde,
        common_norm=False,
        palette=palette,
        ax=ax
    )

    # -------------------
    # Reference line
    # -------------------
    plt.axvline(
        0,
        linestyle='--',
        color='r'
    )

    # -------------------
    # Median lines
    # -------------------
    center_handles = []

    if show_center:

        for level in levels:

            sub = df[df['cesi_level'] == level]

            median_val = sub['coef_Temp'].median()

            label = (
                cesi_label_map.get(level, level)
                if cesi_label_map else level
            )

            plt.axvline(
                median_val,
                linestyle='--',
                linewidth=2,
                color=palette[level],
                alpha=0.9
            )

            center_handles.append(
                mlines.Line2D(
                    [],
                    [],
                    color=palette[level],
                    linestyle='--',
                    linewidth=2,
                    label=f'{label}'
                )
            )

    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Change in Voltage",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    ylabel = (
        "Percentage of Sensors (%)"
        if normalize else "Count"
    )

    plt.ylabel(
        ylabel,
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    # -------------------
    # Percent formatting
    # -------------------
    if normalize:
        ax.yaxis.set_major_formatter(
            PercentFormatter(xmax=100)
        )

    # -------------------
    # Gridlines
    # -------------------
    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    # -------------------
    # Legend
    # -------------------
    handles, labels = ax.get_legend_handles_labels()

    clean_handles = []
    clean_labels = []

    for h, l in zip(handles, labels):

        if l in levels:

            clean_handles.append(h)
            clean_labels.append(l)

    if cesi_label_map is not None:
        clean_labels = [
            cesi_label_map.get(l, l)
            for l in clean_labels
        ]

    clean_handles += center_handles
    clean_labels += [
        h.get_label()
        for h in center_handles
    ]

    zero_handle = mlines.Line2D(
        [],
        [],
        color='red',
        linestyle='--',
        linewidth=2,
        label=zero_line_label
    )

    clean_handles.append(zero_handle)
    clean_labels.append(zero_line_label)

    if len(clean_handles) > 0:

        plt.legend(
            handles=clean_handles,
            labels=clean_labels,
            fontsize=14,
            frameon=True,
            loc=legend_loc,
            bbox_to_anchor=legend_bbox
        )

    # -------------------
    # Ticks
    # -------------------
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_temp_coef_cdf_by_cesi(
    df,
    figsize=(12, 8),
    show_grid_lines=True,
    cesi_label_map=None,
    legend_loc='best',
    legend_bbox=None,
    zero_line_label='No Temperature Effect'
):

    import matplotlib.lines as mlines
    from matplotlib.ticker import PercentFormatter
    import seaborn as sns
    import matplotlib.pyplot as plt

    plt.figure(figsize=figsize)
    ax = plt.gca()

    # -------------------
    # Consistent palette
    # -------------------
    levels = df['cesi_level'].dropna().unique()

    palette = dict(
        zip(levels, sns.color_palette(n_colors=len(levels)))
    )

    # -------------------
    # ECDF
    # -------------------
    sns.ecdfplot(
        data=df,
        x='coef_Temp',
        hue='cesi_level',
        palette=palette,
        ax=ax,
        legend=False
    )

    # -------------------
    # Reference line
    # -------------------
    plt.axvline(
        0,
        linestyle='--',
        color='r'
    )

    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Change in Voltage",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    plt.ylabel(
        "Percentage of Sensors",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    ax.yaxis.set_major_formatter(
        PercentFormatter(xmax=1)
    )

    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    # -------------------
    # Manual legend
    # -------------------
    legend_handles = []

    # CESI lines
    for level in levels:

        label = (
            cesi_label_map.get(level, level)
            if cesi_label_map else level
        )

        legend_handles.append(
            mlines.Line2D(
                [], [],
                color=palette[level],
                linewidth=2,
                label=label
            )
        )

    # Zero-effect line
    legend_handles.append(
        mlines.Line2D(
            [], [],
            color='red',
            linestyle='--',
            linewidth=2,
            label=zero_line_label
        )
    )

    plt.legend(
        handles=legend_handles,
        fontsize=14,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    # -------------------
    # Ticks
    # -------------------
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
## revised function - cmap for CESI 

In [ ]:
def plot_temp_coef_cdf_by_cesi(
    df,
    figsize=(12, 8),
    show_grid_lines=True,
    cesi_label_map=None,
    legend_loc='best',
    legend_bbox=None,
    zero_line_label='No Temperature Effect'
):
    import matplotlib.lines as mlines
    from matplotlib.ticker import PercentFormatter
    import seaborn as sns
    import matplotlib.pyplot as plt

    plt.figure(figsize=figsize)
    ax = plt.gca()

    # -------------------
    # Consistent palette
    # -------------------
    levels = df['cesi_level'].dropna().unique()

    cesi_colors = {
        'Low':    '#6a51a3',
        'Medium': '#238b45',
        'High':   '#8c2d04'
    }

    palette = {
        level: cesi_colors.get(level, fallback)
        for level, fallback in zip(
            levels,
            sns.color_palette(n_colors=len(levels)).as_hex()
        )
    }

    # -------------------
    # ECDF
    # -------------------
    sns.ecdfplot(
        data=df,
        x='coef_Temp',
        hue='cesi_level',
        palette=palette,
        ax=ax,
        legend=False
    )

    # -------------------
    # Reference line
    # -------------------
    plt.axvline(
        0,
        linestyle='--',
        color='r'
    )

    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Change in Voltage",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    plt.ylabel(
        "Percentage of Sensors",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    ax.yaxis.set_major_formatter(
        PercentFormatter(xmax=1)
    )

    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    # -------------------
    # Manual legend
    # -------------------
    legend_handles = []

    for level in levels:
        label = (
            cesi_label_map.get(level, level)
            if cesi_label_map else level
        )
        legend_handles.append(
            mlines.Line2D(
                [], [],
                color=palette[level],
                linewidth=2,
                label=label
            )
        )

    legend_handles.append(
        mlines.Line2D(
            [], [],
            color='red',
            linestyle='--',
            linewidth=2,
            label=zero_line_label
        )
    )

    plt.legend(
        handles=legend_handles,
        fontsize=14,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    # -------------------
    # Ticks
    # -------------------
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
def print_coef_threshold_stats(
    df,
    threshold,
    cesi_label_map=None
):

    levels = df['cesi_level'].dropna().unique()
    results = {}

    for level in levels:
        sub = df[df['cesi_level'] == level]

        if len(sub) == 0:
            pct = np.nan
        else:
            pct = (sub['coef_Temp'] <= threshold).mean() * 100

        label = cesi_label_map.get(level, level) if cesi_label_map else level
        results[label] = pct

    # Pretty print
    print(f"\nAt threshold = {threshold}:\n")
    for label, pct in results.items():
        if not np.isnan(pct):
            print(f"{pct:.1f}% of {label} sensors have coefficients ≤ {threshold}")
        else:
            print(f"No data for {label}")

In [ ]:
def plot_temp_coef_cdf(
    df,
    figsize=(12, 8),
    show_grid_lines=True
):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.ticker import PercentFormatter

    plt.figure(figsize=figsize)

    sns.ecdfplot(
        data=df,
        x='coef_Temp'
    )

    # Reference line at zero
    plt.axvline(0, linestyle='--', color='r')

    # Labels
    plt.xlabel("Change in Voltage", fontsize=16, labelpad=15, fontweight='bold')
    plt.ylabel("Cumulative Percentage of Sensors", fontsize=16, labelpad=15, fontweight='bold')

    ax = plt.gca()
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))

    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

### monthly distribution df 

In [ ]:
distribution_month_df = build_monthly_coef_distribution(per_sensor_complete)

In [ ]:
distribution_month_df.head()

### monthly distribution 

In [ ]:
plot_monthly_distribution(
    distribution_month_df,
    center_stat='median', 
    show_fliers=False,
    ylim=(-25, 25),
    legend_loc='upper left'
)

### Vertically stacked version 

In [ ]:
def plot_month_hour_distribution(
    month_df,
    hour_df,
    month_ylim=None,
    hour_ylim=None,
    center_stat='mean',
    show_fliers=True,
    shade_range=None,
    shade_label=None,
    legend_loc='best',
    legend_bbox=None,
    figsize=(12, 10),
    save_path=None,
    dpi=300
):
    import seaborn as sns
    import matplotlib.pyplot as plt
    import matplotlib.lines as mlines
    import matplotlib.patches as mpatches
    import numpy as np
    import calendar

    fig, (ax_month, ax_hour) = plt.subplots(2, 1, figsize=figsize, sharex=False)

    # -------------------------
    # MONTH PANEL
    # -------------------------
    month_df = month_df.copy()
    month_df['month_label'] = month_df['month'].apply(lambda m: calendar.month_abbr[m])
    order = [calendar.month_abbr[m] for m in range(2, 13)]

    sns.boxplot(
        x='month_label', y='coef',
        data=month_df, order=order,
        showfliers=show_fliers,
        ax=ax_month
    )

    if center_stat is not None:
        if center_stat not in ['mean', 'median']:
            raise ValueError("center_stat must be 'mean', 'median', or None")
        center_df = (
            month_df.groupby('month', as_index=False)['coef']
            .agg(center_stat)
            .sort_values('month')
        )
        ax_month.plot(
            center_df['month'] - 2,
            center_df['coef'],
            marker='o', linewidth=2, color='black'
        )

    ax_month.axhline(0, linestyle='--', color='r', label='Zero effect')
    ax_month.set_xlabel("Month", fontsize=16, labelpad=15, fontweight='bold')
    ax_month.set_ylabel("")
    ax_month.tick_params(axis='x', labelsize=12)
    ax_month.tick_params(axis='y', labelsize=12)

    if month_ylim is not None:
        ax_month.set_ylim(month_ylim)
    else:
        max_val = np.ceil(np.max(np.abs(month_df['coef'])))
        ax_month.set_ylim(-max_val, max_val)

    ax_month.spines['top'].set_visible(False)
    ax_month.spines['right'].set_visible(False)
    ax_month.text(
        -0.07, 1.0, 'a', transform=ax_month.transAxes,
        fontsize=16, fontweight='bold', va='top'
    )

    # -------------------------
    # HOUR PANEL
    # -------------------------
    sns.boxplot(
        x='hour', y='coef',
        data=hour_df,
        showfliers=show_fliers,
        ax=ax_hour
    )

    if center_stat is not None:
        center_df = (
            hour_df.groupby('hour', as_index=False)['coef']
            .agg(center_stat)
            .sort_values('hour')
        )
        ax_hour.plot(
            center_df['hour'] - 1,
            center_df['coef'],
            marker='o', linewidth=2, color='black'
        )

    if shade_range is not None:
        start, end = shade_range
        ax_hour.axvspan(start - 1, end - 1, color='orange', alpha=0.15)

    ax_hour.axhline(0, linestyle='--', color='r', label='_nolegend_')
    ax_hour.set_xlabel("Hour of day", fontsize=16, labelpad=15, fontweight='bold')
    ax_hour.set_ylabel("")
    ax_hour.tick_params(axis='x', labelsize=12)
    ax_hour.tick_params(axis='y', labelsize=12)

    if hour_ylim is not None:
        ax_hour.set_ylim(hour_ylim)
    else:
        max_val = np.ceil(np.max(np.abs(hour_df['coef'])))
        ax_hour.set_ylim(-max_val, max_val)

    ax_hour.spines['top'].set_visible(False)
    ax_hour.spines['right'].set_visible(False)
    ax_hour.text(
        -0.07, 1.0, 'b', transform=ax_hour.transAxes,
        fontsize=16, fontweight='bold', va='top'
    )

    # -------------------------
    # Shared y label
    # -------------------------
    fig.text(
        0.02, 0.5, 'Distribution of Coefficients',
        va='center', ha='center',
        rotation='vertical',
        fontsize=17, fontweight='bold'
    )

    # -------------------------
    # Shared legend
    # -------------------------
    legend_handles = []

    legend_handles.append(
        mlines.Line2D(
            [], [], color='r', linestyle='--',
            linewidth=1.5, label='Zero effect'
        )
    )

    if center_stat is not None:
        legend_handles.append(
            mlines.Line2D(
                [], [], color='black', marker='o',
                linewidth=2,
                label=f'{center_stat.capitalize()} coefficient'
            )
        )

    if shade_range is not None and shade_label is not None:
        legend_handles.append(
            mpatches.Patch(color='orange', alpha=0.3, label=shade_label)
        )

    fig.legend(
        handles=legend_handles,
        fontsize=14,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    # -------------------------
    # Save / show
    # -------------------------
    plt.tight_layout(rect=[0.05, 0, 1, 1])

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')

    plt.show()
    return fig, (ax_month, ax_hour)

In [ ]:
# Usage
fig, axes = plot_month_hour_distribution(
    month_df=distribution_month_df,
    hour_df=distribution_hour_df,
    month_ylim=(-25, 25),
    hour_ylim=(-25, 20),
    center_stat='median',
    show_fliers=False,
    shade_range=(17, 21),
    shade_label="Peak load period",
    legend_loc='upper right',
    legend_bbox=(1.0, 1.1),
    figsize=(12, 10),
    # save_path='fig_month_hour_per_sensor.png'
)

### monthly sensor stats 

In [ ]:
monthly_counts_df = get_monthly_sensor_counts(distribution_month_df)

In [ ]:
monthly_counts_df

### checking for statistical significance by month 

In [ ]:
monthly_test_df = compare_cesi_coefficients_by_month(
    distribution_month_df,
    baseline_level='High',
    comparison_level='Low'
)

In [ ]:
# monthly_test_df

### Delta Median (Low - High CESI) Plot 

In [ ]:
monthly_delta_df = plot_monthly_median_delta(
    distribution_month_df,
    test_df=monthly_test_df,
    baseline_level='High',
    comparison_level='Low',
    annotate=False,
    ylim=(-5, 5),
    legend_loc='upper left',
    legend_bbox=(0.80, 1),
    ylabel='Delta Median Coefficient'
)

### `positive values`: Means the Low CESI areas had higher voltages / High CESI had lower voltages 

### `negative values`: Means the Low CESI areas had lower voltages / High CESI had higher voltages 

#### however apart from April, none of the differences were statistically significant 

#### that swing in Low CESI is interesting: `Higher voltages` in April/May & `Lower voltages` in Feb/December 

#### if we're looking at it in terms of rainy vs dry season, the trend doesn't appear to be consistent 

#### these are all with `reference` to January though (but still here, I'm comparing the 2) 

In [ ]:
def plot_month_hour_median_delta(
    month_df,
    hour_df,
    month_test_df=None,
    hour_test_df=None,
    baseline_level='High',
    comparison_level='Low',
    month_ylim=None,
    hour_ylim=None,
    color='firebrick',
    marker='o',
    linewidth=3,
    linestyle='-',
    show_grid_lines=True,
    annotate=False,
    annotate_sig=True,
    sig_y_offset=0.25,
    sig_fontsize=16,
    show_sig_legend=True,
    shade_range=None,
    shade_label=None,
    legend_loc='best',
    legend_bbox=None,
    ylabel='Delta Median Coefficient',
    figsize=(12, 10),
    save_path=None,
    dpi=300
):
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import matplotlib.lines as mlines
    import calendar

    fig, (ax_month, ax_hour) = plt.subplots(2, 1, figsize=figsize, sharex=False)

    # -------------------------
    # MONTH PANEL
    # -------------------------
    month_df = month_df.copy()
    median_month = (
        month_df.groupby(['month', 'cesi_level'])['coef']
        .median()
        .reset_index()
    )
    pivot_month = median_month.pivot(
        index='month', columns='cesi_level', values='coef'
    )
    pivot_month['delta'] = pivot_month[comparison_level] - pivot_month[baseline_level]
    pivot_month = pivot_month.reset_index()
    pivot_month['month_label'] = pivot_month['month'].apply(
        lambda m: calendar.month_abbr[m]
    )

    ax_month.plot(
        pivot_month['month_label'], pivot_month['delta'],
        color=color, marker=marker, linewidth=linewidth, linestyle=linestyle,
        label=f'{comparison_level} - {baseline_level}'
    )
    ax_month.axhline(0, linestyle='--', color='black', linewidth=1.5, label='No difference')

    if annotate:
        for _, row in pivot_month.iterrows():
            ax_month.text(
                row['month_label'], row['delta'],
                f"{row['delta']:.2f}", fontsize=10, ha='center', va='bottom'
            )

    if month_test_df is not None and annotate_sig:
        sig_df = month_test_df[
            (month_test_df['significant'] == True) &
            (month_test_df['sig_stars'].notna()) &
            (month_test_df['sig_stars'] != '')
        ]
        sig_map = dict(zip(sig_df['month'], sig_df['sig_stars']))
        for _, row in pivot_month.iterrows():
            if row['month'] in sig_map:
                y = row['delta']
                offset = sig_y_offset if y >= 0 else -sig_y_offset
                ax_month.text(
                    row['month_label'], y + offset,
                    sig_map[row['month']],
                    fontsize=sig_fontsize, fontweight='bold',
                    ha='center', va='bottom' if y >= 0 else 'top'
                )

    ax_month.set_xlabel("Month", fontsize=16, labelpad=15, fontweight='bold')
    ax_month.set_ylabel("")
    ax_month.tick_params(axis='x', labelsize=12)
    ax_month.tick_params(axis='y', labelsize=12)

    if month_ylim is not None:
        ax_month.set_ylim(month_ylim)
    else:
        max_val = np.ceil(np.max(np.abs(pivot_month['delta'])))
        ax_month.set_ylim(-max_val, max_val)

    if show_grid_lines:
        ax_month.grid(axis='y', linestyle='--', alpha=0.3)

    ax_month.spines['top'].set_visible(False)
    ax_month.spines['right'].set_visible(False)
    ax_month.text(
        -0.07, 1.0, 'a', transform=ax_month.transAxes,
        fontsize=16, fontweight='bold', va='top'
    )

    # -------------------------
    # HOUR PANEL
    # -------------------------
    hour_df = hour_df.copy()
    median_hour = (
        hour_df.groupby(['hour', 'cesi_level'])['coef']
        .median()
        .reset_index()
    )
    pivot_hour = median_hour.pivot(
        index='hour', columns='cesi_level', values='coef'
    )
    pivot_hour['delta'] = pivot_hour[comparison_level] - pivot_hour[baseline_level]
    pivot_hour = pivot_hour.reset_index().sort_values('hour')

    if shade_range is not None:
        start, end = shade_range
        ax_hour.axvspan(start, end, color='orange', alpha=0.15)

    ax_hour.plot(
        pivot_hour['hour'], pivot_hour['delta'],
        color=color, marker=marker, linewidth=linewidth, linestyle=linestyle,
        label=f'{comparison_level} - {baseline_level}'
    )
    ax_hour.axhline(0, linestyle='--', color='black', linewidth=1.5, label='_nolegend_')

    if annotate:
        for _, row in pivot_hour.iterrows():
            ax_hour.text(
                row['hour'], row['delta'],
                f"{row['delta']:.2f}", fontsize=9, ha='center', va='bottom'
            )

    if hour_test_df is not None and annotate_sig:
        sig_df = hour_test_df[
            (hour_test_df['significant'] == True) &
            (hour_test_df['sig_stars'].notna()) &
            (hour_test_df['sig_stars'] != '')
        ]
        sig_map = dict(zip(sig_df['hour'], sig_df['sig_stars']))
        for _, row in pivot_hour.iterrows():
            if row['hour'] in sig_map:
                y = row['delta']
                offset = sig_y_offset if y >= 0 else -sig_y_offset
                ax_hour.text(
                    row['hour'], y + offset,
                    sig_map[row['hour']],
                    fontsize=sig_fontsize, fontweight='bold',
                    ha='center', va='bottom' if y >= 0 else 'top'
                )

    ax_hour.set_xlabel("Hour of day", fontsize=16, labelpad=15, fontweight='bold')
    ax_hour.set_ylabel("")
    ax_hour.set_xticks(range(0, 24))
    ax_hour.tick_params(axis='x', labelsize=12)
    ax_hour.tick_params(axis='y', labelsize=12)

    if hour_ylim is not None:
        ax_hour.set_ylim(hour_ylim)
    else:
        max_val = np.ceil(np.max(np.abs(pivot_hour['delta'])))
        ax_hour.set_ylim(-max_val, max_val)

    if show_grid_lines:
        ax_hour.grid(axis='y', linestyle='--', alpha=0.3)

    ax_hour.spines['top'].set_visible(False)
    ax_hour.spines['right'].set_visible(False)
    ax_hour.text(
        -0.07, 1.0, 'b', transform=ax_hour.transAxes,
        fontsize=16, fontweight='bold', va='top'
    )

    # -------------------------
    # Shared y label
    # -------------------------
    fig.text(
        0.02, 0.5, ylabel,
        va='center', ha='center',
        rotation='vertical',
        fontsize=16, fontweight='bold'
    )

    # -------------------------
    # Shared legend
    # -------------------------
    legend_handles = []

    legend_handles.append(
        mlines.Line2D(
            [], [], color=color, marker=marker,
            linewidth=linewidth, linestyle=linestyle,
            label=f'{comparison_level} - {baseline_level}'
        )
    )

    legend_handles.append(
        mlines.Line2D(
            [], [], color='black', linestyle='--',
            linewidth=1.5, label='No difference'
        )
    )

    if shade_range is not None and shade_label is not None:
        legend_handles.append(
            mpatches.Patch(color='orange', alpha=0.3, label=shade_label)
        )

    if show_sig_legend:
        legend_handles.append(
            mlines.Line2D(
                [], [], color='black', marker='$*$',
                linestyle='None', markersize=6, label='p < 0.05'
            )
        )
        legend_handles.append(
            mlines.Line2D(
                [], [], color='black', marker='$**$',
                linestyle='None', markersize=14, label='p < 0.01'
            )
        )

    fig.legend(
        handles=legend_handles,
        fontsize=14,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    # -------------------------
    # Save / show
    # -------------------------
    plt.tight_layout(rect=[0.05, 0, 1, 1])

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')

    plt.show()
    return pivot_month, pivot_hour

In [ ]:
# Usage
pivot_month, pivot_hour = plot_month_hour_median_delta(
    month_df=distribution_month_df,
    hour_df=distribution_hour_df,
    month_test_df=monthly_test_df,
    hour_test_df=hourly_test_df,
    baseline_level='High',
    comparison_level='Low',
    month_ylim=(-5, 5),
    hour_ylim=(-5, 5),
    shade_range=(17, 21),
    shade_label="Peak load period",
    annotate=False,
    legend_loc='upper right',
    legend_bbox=(1.0, 1.05),
    ylabel='Difference in Median Coefficients: Low – High CESI',
    figsize=(12, 10),
    # save_path='fig_month_hour_delta.png'
)

<div style="
    background-color:#12f391;
    color:#664d03;
    padding:12px;
    border-left:5px solid #ffec99;
    border-radius:6px;
    font-size:22px;
    font-weight:bold;">

3) Temperature Co-efficients 

</div>

### keep only the sensors that have significant coefficient with Temp 

In [ ]:
temp_sig_df = per_sensor_complete[per_sensor_complete['sig_Temp'] == True].copy().reset_index(drop=True)

## *** Per Sensor Results 

In [ ]:
def plot_temp_coef_histogram(
    df,
    bins=20,
    figsize=(8, 5),
    show_kde=True,
    normalize=False,
    show_center=True,
    show_grid_lines=True,
    # --- Zero reference line ---
    zero_line_value=0,
    zero_line_label='Zero Coefficient',
    zero_line_color='red',
    zero_line_style='--',
    # --- Overall coefficient line ---
    overall_temp_coeff=None,
    overall_temp_coeff_label='Overall Temp Coeff',
    overall_temp_coeff_color='darkorange',
    overall_temp_coeff_style='--',
    # --- Legend ---
    legend_loc='best',
    legend_bbox=None,
    # --- Save ---
    save_path=None,
    dpi=300
):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.ticker import PercentFormatter

    plt.figure(figsize=figsize)

    # -------------------
    # Histogram
    # -------------------
    if normalize:
        sns.histplot(
            df['coef_Temp'],
            bins=bins,
            kde=show_kde,
            stat='percent',
            color='grey'
        )
    else:
        sns.histplot(
            df['coef_Temp'],
            bins=bins,
            kde=show_kde,
            stat='count',
            color='grey'
        )

    # -------------------
    # Zero reference line
    # -------------------
    plt.axvline(
        zero_line_value,
        linestyle=zero_line_style,
        color=zero_line_color,
        linewidth=2,
        label=zero_line_label
    )

    # -------------------
    # Mean / Median
    # -------------------
    if show_center:
        mean_val = df['coef_Temp'].mean()
        median_val = df['coef_Temp'].median()
        plt.axvline(
            mean_val,
            color='black',
            linestyle='-',
            linewidth=2,
            label='Mean'
        )
        plt.axvline(
            median_val,
            color='black',
            linestyle='--',
            linewidth=2,
            label='Median'
        )

    # -------------------
    # Overall temp coefficient line
    # -------------------
    if overall_temp_coeff is not None:
        plt.axvline(
            overall_temp_coeff,
            color=overall_temp_coeff_color,
            linestyle=overall_temp_coeff_style,
            linewidth=2,
            label=overall_temp_coeff_label
        )

    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Voltage Sensitivity to Temperature (ΔV/°C)",
        fontsize=14,
        labelpad=15,
        fontweight='bold'
    )
    ylabel = "Proportion of Sensors (%)" if normalize else "Count"
    plt.ylabel(
        ylabel,
        fontsize=14,
        labelpad=15,
        fontweight='bold'
    )

    ax = plt.gca()

    # -------------------
    # Percent formatting
    # -------------------
    if normalize:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))

    # -------------------
    # Gridlines
    # -------------------
    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    # -------------------
    # Ticks
    # -------------------
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # -------------------
    # Legend (orange line first)
    # -------------------
    handles, labels = ax.get_legend_handles_labels()
    if overall_temp_coeff is not None and overall_temp_coeff_label in labels:
        idx = labels.index(overall_temp_coeff_label)
        handles = [handles[idx]] + handles[:idx] + handles[idx+1:]
        labels = [labels[idx]] + labels[:idx] + labels[idx+1:]

    plt.legend(
        handles=handles,
        labels=labels,
        frameon=False,
        fontsize=12,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')

    plt.show()

In [ ]:
plot_temp_coef_histogram(
    temp_sig_df,
    bins = 40, 
    normalize = True,
    show_kde = True, 
    show_center = False, 
    show_grid_lines = True,
    overall_temp_coeff = -1.05,
    overall_temp_coeff_label = 'Pooled-Model Sensitivity', 
    zero_line_label = 'Zero Sensitivity',
    legend_loc = 'upper left',
    legend_bbox = (0.55, 1.20), 
    figsize=(9, 6),
    # save_path=PLOTS_DIR / 'histogram_per_sensor_coeff.png'
)

In [ ]:
temp_sig_df[(temp_sig_df['sig_Temp']==True) & (temp_sig_df['coef_Temp']<=0)]

In [ ]:
temp_sig_df

In [ ]:
def plot_temp_coef_histogram_by_cesi(
    df,
    bins=20,
    figsize=(8, 5),
    normalize=True,
    show_center=True,
    show_kde=True,
    show_grid_lines=True,
    fill_value=True,
    cesi_label_map=None,
    legend_loc='best',
    legend_bbox=None,
    zero_line_label='No Temperature Effect',
    level_order=None,
    cesi_color_map=None,
    save_path=None
):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.ticker import PercentFormatter
    import matplotlib.lines as mlines

    plt.figure(figsize=figsize)
    ax = plt.gca()

    stat = 'percent' if normalize else 'count'

    # -------------------
    # Consistent palette
    # -------------------
    levels = df['cesi_level'].dropna().unique()

    if level_order is not None:
        levels = [lvl for lvl in level_order if lvl in levels]

    # Fixed color mapping (High = orange, Low = blue), with fallback to defaults
    default_colors = {'Low': 'tab:blue', 'High': 'tab:orange'}
    color_map = cesi_color_map if cesi_color_map is not None else default_colors

    palette = {
        lvl: color_map.get(lvl, c)
        for lvl, c in zip(levels, sns.color_palette(n_colors=len(levels)))
    }

    # -------------------
    # Histogram + KDE
    # -------------------
    sns.histplot(
        data=df,
        x='coef_Temp',
        hue='cesi_level',
        bins=bins,
        stat=stat,
        element='step',
        fill=fill_value,
        alpha=0.2,
        kde=show_kde,
        common_norm=False,
        palette=palette,
        ax=ax
    )

    # -------------------
    # Reference line
    # -------------------
    plt.axvline(
        0,
        linestyle='--',
        color='r'
    )

    # -------------------
    # Median lines
    # -------------------
    center_handles = []

    if show_center:

        for level in levels:

            sub = df[df['cesi_level'] == level]

            median_val = sub['coef_Temp'].median()

            label = (
                cesi_label_map.get(level, level)
                if cesi_label_map else level
            )

            plt.axvline(
                median_val,
                linestyle='--',
                linewidth=2,
                color=palette[level],
                alpha=0.9
            )

            center_handles.append(
                mlines.Line2D(
                    [],
                    [],
                    color=palette[level],
                    linestyle='--',
                    linewidth=2,
                    label=f'{label}'
                )
            )

    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Voltage Sensitivity to Temperature (ΔV/°C)",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    ylabel = (
        "Proportion of Sensors (%)"
        if normalize else "Count"
    )

    plt.ylabel(
        ylabel,
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )

    # -------------------
    # Percent formatting
    # -------------------
    if normalize:
        ax.yaxis.set_major_formatter(
            PercentFormatter(xmax=100)
        )

    # -------------------
    # Gridlines
    # -------------------
    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    # -------------------
    # Legend
    # -------------------
    handles, labels = ax.get_legend_handles_labels()

    clean_handles = []
    clean_labels = []

    for h, l in zip(handles, labels):

        if l in levels:

            clean_handles.append(h)
            clean_labels.append(l)

    # Apply CESI label map
    if cesi_label_map is not None:
        clean_labels = [
            cesi_label_map.get(l, l)
            for l in clean_labels
        ]

    # Add center handles
    clean_handles += center_handles
    clean_labels += [
        h.get_label()
        for h in center_handles
    ]

    # Add zero-effect line to legend
    zero_handle = mlines.Line2D(
        [],
        [],
        color='red',
        linestyle='--',
        linewidth=2,
        label=zero_line_label
    )

    clean_handles.append(zero_handle)
    clean_labels.append(zero_line_label)

    if len(clean_handles) > 0:

        plt.legend(
            handles=clean_handles,
            labels=clean_labels,
            fontsize=12,
            frameon=True,
            loc=legend_loc,
            bbox_to_anchor=legend_bbox
        )

    # -------------------
    # Ticks
    # -------------------
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()

    # -------------------
    # Save figure
    # -------------------
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
plot_temp_coef_histogram_by_cesi(
    temp_sig_df,
    figsize=(10, 7),
    show_kde=True,
    show_center=True,
    cesi_label_map={
        'Low': 'Median Sensitivity (Low CESI)',
        'High': 'Median Sensitivity (High CESI)'
    },
    legend_loc='upper left',
    legend_bbox=(0.55, 1.22),
    zero_line_label='Zero Sensitivity',
    level_order=['High', 'Low'],
    save_path = PLOTS_DIR / 'sensor_voltage_cesi_pdfs.png'
)

In [ ]:
def plot_temp_coef_cdf_by_cesi(
    df,
    figsize=(12, 8),
    show_grid_lines=True,
    cesi_label_map=None,
    legend_loc='best',
    legend_bbox=None,
    zero_line_label='No Temperature Effect',
    save_path=None,
    ylim=None
):
    import matplotlib.lines as mlines
    from matplotlib.ticker import PercentFormatter
    import seaborn as sns
    import matplotlib.pyplot as plt
    plt.figure(figsize=figsize)
    ax = plt.gca()
    # -------------------
    # Consistent palette
    # -------------------
    levels = df['cesi_level'].dropna().unique()
    palette = dict(
        zip(levels, sns.color_palette(n_colors=len(levels)))
    )
    # -------------------
    # ECDF
    # -------------------
    sns.ecdfplot(
        data=df,
        x='coef_Temp',
        hue='cesi_level',
        palette=palette,
        ax=ax,
        legend=False
    )
    # -------------------
    # Reference line
    # -------------------
    plt.axvline(
        0,
        linestyle='--',
        color='r'
    )
    # -------------------
    # Labels
    # -------------------
    plt.xlabel(
        "Voltage Sensitivity to Temperature (ΔV/°C)",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )
    plt.ylabel(
        "Proportion of Sensors (%)",
        fontsize=16,
        labelpad=15,
        fontweight='bold'
    )
    ax.yaxis.set_major_formatter(
        PercentFormatter(xmax=1)
    )
    # -------------------
    # Y-axis limits
    # -------------------
    if ylim is not None:
        ax.set_ylim(ylim)
    if show_grid_lines:
        ax.grid(axis='y', linestyle='--', alpha=0.3)
    # -------------------
    # Manual legend
    # -------------------
    legend_handles = []
    # CESI lines
    for level in levels:
        label = (
            cesi_label_map.get(level, level)
            if cesi_label_map else level
        )
        legend_handles.append(
            mlines.Line2D(
                [], [],
                color=palette[level],
                linewidth=2,
                label=label
            )
        )
    # Zero-effect line
    legend_handles.append(
        mlines.Line2D(
            [], [],
            color='red',
            linestyle='--',
            linewidth=2,
            label=zero_line_label
        )
    )
    plt.legend(
        handles=legend_handles,
        fontsize=12,
        frameon=True,
        loc=legend_loc,
        bbox_to_anchor=legend_bbox
    )
    # -------------------
    # Ticks
    # -------------------
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    # -------------------
    # Remove spines
    # -------------------
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    # -------------------
    # Save figure
    # -------------------
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_temp_coef_cdf_by_cesi(
    temp_sig_df,
    figsize=(9, 7),
    ylim=(-0.05, 1), 
    cesi_label_map={
        'Low': 'Low CESI',
        'High': 'High CESI'
    }, 
    legend_loc = 'upper left',
    legend_bbox = (0.68, 1.22), 
    zero_line_label = 'Zero Sensitivity',
    save_path = PLOTS_DIR / 'sensor_voltage_cesi_cdfs.png'
)

In [ ]:
temp_sig_df[['coef_Temp', 'cesi_level']].groupby('cesi_level')[['coef_Temp']].describe()

In [ ]:
print_coef_threshold_stats(
    temp_sig_df,
    threshold = -1.0,
    cesi_label_map={
        'Low': 'Low CESI',
        'High': 'High CESI'
    }
)

In [ ]:
plot_temp_coef_cdf(temp_sig_df)

In [ ]:
pct_negative = (temp_sig_df['coef_Temp'] < 0).mean() * 100
print(f"{pct_negative:.1f}% of significant sensors have negative temperature coefficients")

### this can be said in text 

### save to file 

In [ ]:
# temp_sig_df.copy()[['site_id', 'sensor_id', 'coef_Temp', 'sig_Temp', 'cesi_level']].to_csv('ols_per_sensor_results.csv')

In [ ]:
temp_sig_df.columns

In [ ]:
len(temp_sig_df)

### Check for if the diff in distributions of per sensor OLS (High & Low CESI) is statistically significant 

In [ ]:
df = temp_sig_df[['site_id', 'sensor_id', 'coef_Temp', 'cesi_level']].copy()

In [ ]:
from scipy.stats import mannwhitneyu

low = df[df['cesi_level'] == 'Low']['coef_Temp']
high = df[df['cesi_level'] == 'High']['coef_Temp']

stat, p = mannwhitneyu(
    low,
    high,
    alternative='two-sided'
)

print("U statistic:", stat)
print("p-value:", p)

In [ ]:
print("Low median:", low.median())
print("High median:", high.median())

In [ ]:
n1 = len(low)
n2 = len(high)

rbc = 1 - (2 * stat) / (n1 * n2)

print("Rank-biserial correlation:", rbc)

In [ ]:
# The distribution of temperature coefficients differed between Low and High CESI sensors, with High CESI sensors exhibiting systematically more negative coefficients (rank-biserial correlation = −0.30), indicating stronger voltage sensitivity to elevated temperatures in climate-vulnerable areas.

#### Temperature coefficient distributions differed significantly between Low and High CESI sensors (Mann–Whitney U = 16,757, p < 0.001). 

#### High CESI sensors exhibited systematically more negative coefficients, indicating stronger voltage sensitivity to elevated temperatures in more climate-vulnerable neighborhoods (rank-biserial correlation = −0.30).

#### the rank-biserial correlation is an effect size measure for rank-based tests like the Mann–Whitney U test 

#### It tells you how separated the two groups are 

#### Median temperature coefficients were −1.02 V/°C for Low CESI sensors and −1.42 V/°C for High CESI sensors.